# 🧪 Core Vision Perfect V1 — 09 · CHIẾN DỊCH MỘT LẦN (round-88 → round-96)

**Mục đích:** đo *mọi ý tưởng còn lại* trong **một phiên, một Run all** — 22 "cánh"
thí nghiệm chạy nối tiếp, mỗi cánh = đúng dàn vũ khí trận (gói ABK) + **một thay đổi
duy nhất**, nên công/tội quy được cho từng ý tưởng. Kết quả từng cánh lưu riêng lên
Drive ngay khi xong; chạy lại notebook thì cánh đã có kết quả được **bỏ qua**.

| # | Cánh | Thay đổi duy nhất | Trả lời câu hỏi |
|---|---|---|---|
| 1 | `TUNE` | dò lại trọng số fusion trên tower mới (xuất phát từ mặc định, + cross-validation 16 fold) | ghi `best_weights-candidate.json` (KHÔNG đụng file trận) |
| 2 | `ABK` | không (đo lại cùng phiên) | thước đo nhiễu so với bench ABK cũ trên Drive |
| 3 | `ABK+TUNED` | ABK + trọng số ứng viên (shrinkage 50%) | trọng số mới có thắng? (tự bỏ nếu ứng viên ≡ trận hoặc tuner không thắng in-sample) |
| 4 | `ABK+W` | trọng số OCR/ASR theo câu hỏi (heuristic) | câu trích chữ/lời có được lợi? |
| 5 | `ABK+RRF` | fusion RRF thay weighted_sum | chưa từng đo với dàn hiện tại |
| 6 | `DIVERSE` | 3 lane 45/30/25 + qwen_embed, ĐẦY ĐỦ reranker | lượt nộp 2 chạy nổi trên A100-40GB? điểm bao nhiêu? |
| 7 | `ABK+V5` | VLM rerank 5 phiếu | tách riêng khỏi gói X |
| 8 | `MERGE2` | trộn RRF `ABK ⊕ DIVERSE` (hedge QA) | kế hoạch lượt 2 THẬT được bao nhiêu? |
| 9 | `MERGE3` | trộn RRF `ABK ⊕ DIVERSE ⊕ ABK+V5` | máy thứ 3 có đáng không? |
| 10 | `MERGE_SIB` | đối chứng: trộn `ABK ⊕ ABK+V5` (cùng đội hình) | phần "được" của trộn chỉ là nhiễu? |
| 11 | `MERGE2_NOHEDGE` | đối chứng: MERGE2 không hedge QA | hedge QA đóng góp bao nhiêu? |
| 12 | `ABK+G38R` | VLM rerank bằng **Gemini 3.8 Flash** (GA 02/09/2026) thay 3.5-flash-lite | model mới có xếp lại top-48 tốt hơn? |
| 13 | `ABK+G38QA` | QA trả lời bằng **Gemini 3.8 Flash** thay 3.1-pro-preview | rẻ 3-5×, ít bão hơn — có giữ được điểm QA? |
| 14 | `ABK+BREAKER` | cầu dao bão Gemini BẬT (round-96) | API khỏe thì phải = ABK (bằng chứng "không đổi một byte") |
| 15 | `ABK+OCRCTX` | QA đọc thêm chữ OCR của các khung trong strip (cùng ASR) | tên trường/địa danh/con số trong chữ chạy có cứu QA? |
| 16 | `ABK+F6` | strip QA 6 khung thay 3 | chữ trải nhiều khung có đọc trọn hơn? |
| 17 | `ABK+LOCALR` | VLM rerank bằng **VLM cục bộ** (`LOCAL_VLM_ID`) thay Gemini | plan B khi Gemini bão đáng tin đến đâu? |
| 18 | `ABK+LOCALQA` | QA trả lời bằng **VLM cục bộ** thay Gemini Pro | cùng câu hỏi cho QA |
| 19 | `ABK+LOCALR2` / 20 `ABK+LOCALQA2` | như 17/18 với VLM cục bộ thứ hai (`LOCAL_VLM_ID_2`, Qwen3.5-9B) | model nào đáng làm plan B? |
| 21 | `ABK+HFQA` | QA qua **Hugging Face Inference Providers** (`HF_ROUTER_QA_MODEL`, cần secret `HF_TOKEN`) | nhà cung cấp độc lập có giữ điểm QA? |
| 22 | `ABK+HFR` | VLM rerank qua HF Inference Providers (`HF_ROUTER_RERANK_MODEL`) | lane rerank độc lập xếp top-48 ra sao? |

**Thứ tự** đã sắp để các cánh chỉ-đổi-retrieval đứng sát baseline trong cùng cửa sổ
quota Gemini; cánh ngốn Gemini (V5) và nặng GPU (DIVERSE) chạy sau. 4 cánh trộn chạy
offline vài giây.

**Luật lưu (audit r88):** cánh chạy **suy thoái** — OOM, rớt lane, reranker không build,
câu không có CSV, VLM rerank rớt trên >¼ số câu, cánh "đo model X" mà X **tự trả lời dưới 75%**
cuộc gọi (đếm từ log HTTP theo model; X còn phải qua tiền kiểm, thử lại 6 lần khi bão 503/429;
từ 75% đến 90% thì lưu nhưng gắn cờ "rớt model" và không xét thắng) — **KHÔNG được lưu** (in ❌ rồi chạy tiếp cánh sau; chạy lại notebook
sẽ đo lại cánh đó). Mỗi cánh ghi kèm: số bão 429/503, số suy thoái nhẹ, VRAM đỉnh,
dấu phiên, giờ bắt đầu/kết thúc, trọng số dùng.

**Phán quyết (tự tính ở cuối):** thắng khi Δ ≥ max(2×nhiễu giữa các lần đo ABK,
2×bậc điểm 0.2/N) **và** thắng ròng ≥ 2 câu so với mọi lần đo ABK. Cánh đo ở phiên
khác với ABK được so với lần đo ABK **cùng phiên với nó** (ABK hoặc `ABK-prev*`); không có thì
bị đánh dấu "≠ phiên" và không được xét thắng. `ABK+TUNED` còn phải không
thua **mặc định** ở held-out (thua = overfit 23 câu; trọng số trận đã khớp cả 23 câu nên
chỉ in để tham khảo).

**Thời gian:** lần đầu 8 cánh bench × ~60-85′ + dò ~30′ ≈ **9-11 giờ** một A100. Phiên round-96 (các cánh cũ đã có trên Drive): ABK đo lại + BREAKER + OCRCTX + F6 + G38QA + G38R + HFQA + HFR + 4 cánh cục bộ ≈ 12 cánh × 30-45′ ≈ **6-9 giờ** khi Google yên; cánh cục bộ nạp thêm 16-18 GiB model lần đầu.
Chạy qua đêm là vừa. **Chạy lại ở phiên sau** (vd thêm cánh mới): cánh đã xong bị bỏ qua,
nhưng nếu còn cánh bench chưa đo thì **ABK được đo lại** trong phiên đó (baseline phải cùng
phiên; bản ABK cũ thành `ABK-prev*.json` = thêm một lần đo nhiễu). Khuyến nghị **một phiên**; nếu buộc chia, phiên 2 dùng
`ARMS = ["DIVERSE", "ABK+V5"]` rồi chạy lại với `ARMS = "all"` để trộn — các cánh
phiên 2 sẽ mang dấu "≠ phiên" (chỉ so tương đối).

**Quota:** khóa của K lên **Tier 2** từ 04/09 12:24 (`gemini-3.1-pro-preview` 1.000 RPM /
50.000 RPD; Flash 3.x 2.000 RPM / 100K RPD; Flash-Lite 10K RPM / 350K RPD) — một chiến dịch
≈ 300-400 cuộc gọi Pro, không còn là ràng buộc. Cánh nào gặp 429 "exceeded your current quota"
vẫn bị gắn cờ "hết quota" và không được xét thắng. **Bão 503/504 phía Google** (04/09: 80-90 %
cuộc gọi suốt 8 giờ) mới là rủi ro: cánh "đo model X" chỉ được lưu khi X tự trả lời ≥ 75 %;
hai cánh cục bộ (`LOCAL*`) chỉ chạy khi điền `LOCAL_VLM_ID` (id HF đã kiểm chứng) ở cell C1 và
bị bỏ (không lưu) nếu model không nạp được / QA vẫn gọi Gemini Pro.

**Zero-edit:** upload + bật 2 secret (`GITHUB_TOKEN`, `GEMINI_API_KEY`) + Run all.
Tùy chọn nhưng **rất nên** (round-90): thêm secret `GEMINI_API_KEY_2` (và `_3`) là khóa của
một dự án Google Cloud **có billing** khác — phiên 03/09 đốt hết quota ngày của
`gemini-3.1-pro-preview` sau 2 cánh (253 lỗi 429) và QA = 0 suốt phần còn lại; có khóa dự
phòng thì mọi cuộc gọi tự chuyển sang khóa kế khi khóa đang dùng hết quota.
Toggle: `ARMS`, `LOCAL_VLM_ID`, `LOCAL_VLM_ID_2`, `HF_ROUTER_QA_MODEL`, `HF_ROUTER_RERANK_MODEL` ở
cell C1 (đã điền id kiểm chứng 05/09). Secret tùy chọn: `HF_TOKEN` (hai cánh HF; không có thì bỏ),
`GEMINI_VERTEX_KEY` (lane Vertex tự chèn sau model chính ở MỌI cánh — chỉ bật khi đã quyết dùng
cho trận, vì nó đổi chuỗi cứu viện của baseline). Thiếu khóa Gemini → dừng ngay, không chạy 9 giờ sai.

**Drive (khai báo trước, không xóa gì):** ghi `artifacts/lab/campaign/` gồm
`<cánh>_run/` (CSV; khi đo lại, bản cũ xoay sang `<cánh>_run-prev*/`), `<cánh>.json`,
`ABK-prev*.json` (ABK của phiên trước), `best_weights-candidate.json`,
`campaign_summary.json` + `campaign_summary-<phiên>.json`; L1 dựng lại `queries/gt-thunghiem.json` như nb04.
Chỉ **đọc**: `tuning/best_weights.json`, `lab/bench_full.json`,
`lab/bench_full-prev.json`. Chỉ ghi máy ảo: `signal_dumps/campaign`,
`submissions/camp_*`. nb03 không bị ảnh hưởng.

**Gửi lại Claude:** `lab/campaign/campaign_summary.json` + toàn bộ output cell C1
(bảng tổng kết, ma trận điểm từng câu, cột bão/suy thoái/VRAM từng cánh).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1 · PARAMS — the ONLY cell you may need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════╝
DRIVE_PROJECT_DIR = "AIC2025"        # MyDrive/<this>/{data, artifacts}
FIRST_TIME_SETUP = False             # True CHỈ cho lần ĐẦU TIÊN tạo dự án trên
#   Drive trống. Mặc định False: mount "lười metadata" sẽ KHÔNG BAO GIỜ được
#   tự đẻ thư mục dự án sinh đôi nữa (round-66 — bài học 07 live).
REPO_URL  = "https://github.com/ledinhminhquan/Core-Vision_Perfect_V1.git"
REPO_REF  = "main"

# Which dense encoders to build indexes for (order = ensemble order).
#   "siglip2"          multilingual default (needed for training too)
#   "openclip"         English lane (DFN5B ViT-H/14-378) — strongest with translation
#   "qwen_embed"       optional HEAVY lane (Qwen embedding tower — strong, slow)
#   "provided_clip32"  organiser features — instant, no GPU (L-batches only);
#                      auto-added in the catalog cell when clip-features-32 exists
EMBED_MODELS = ["siglip2", "openclip"]

# Copy keyframes from Drive → local disk before embedding (much faster I/O).
COPY_KEYFRAMES_LOCAL = True

# Aux indexes to build (each is resumable; captions are the slowest).
RUN_OCR, RUN_ASR, RUN_CAPTIONS = True, True, True
CAPTION_STRIDE = 4                   # caption mỗi keyframe thứ 4 (round-16: đủ dày
#   cho kênh recall BM25 mà nhanh gấp đôi stride 2. NÂNG stride luôn an toàn với
#   resume: video đã caption ở stride nhỏ hơn vẫn được tính là XONG ở stride lớn
#   hơn. Mọi phiên chạy song song PHẢI dùng CÙNG một stride.

# K-batch shot detection: install TransNetV2 (the winning-team detector) for
# keyframe self-extraction. Installed --no-deps (Colab torch is never touched);
# without it extraction falls back to PySceneDetect automatically. (nb01 only)
INSTALL_TRANSNETV2 = True

# Force-rebuild toggles — mặc định False = resume/skip khi artifact đã có.
FORCE_CATALOG    = False             # rebuild the catalog parquet
FORCE_EMBED      = False             # re-embed every keyframe
FORCE_INDEX      = False             # rebuild the FAISS indexes
FORCE_AUX        = False             # redo OCR/ASR/captions from scratch
FORCE_TEXT_INDEX = False             # rebuild the persisted BM25 text index

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("params ok")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2 · Mount Drive + folder layout + preflight write test          ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, shutil, subprocess, time
from pathlib import Path

from google.colab import drive

_MP = "/content/drive"

def _drive_alive() -> bool:
    try:
        return Path(_MP, "MyDrive").exists()
    except OSError:
        return False

def _ensure_drive():
    """Mount / HỒI SINH Drive FUSE — dùng ở mọi cell dài hơi phía sau.

    Round-19 (live run 8): daemon DriveFS chết để lại mountpoint 'bẩn' →
    drive.mount kêu 'Mountpoint must not already contain files' và cả
    force_remount cũng bó tay. Trình tự cứu đúng: (1) fusermount -uz gỡ
    mount chết; (2) CHỈ khi chắc chắn không còn mount (os.path.ismount ==
    False — lúc này các entry trong mountpoint là RÁC LOCAL trên đĩa VM,
    không phải Drive thật) mới dọn sạch chúng; (3) mount lại.
    """
    for _try in range(4):
        if _drive_alive():
            return
        if _try:
            print(f"⚠ Drive FUSE chưa sống — hồi sinh (lần {_try}/3) ...")
        try:
            if os.path.ismount(_MP):
                subprocess.run(["fusermount", "-uz", _MP], capture_output=True)
                time.sleep(2)
            if os.path.isdir(_MP) and not os.path.ismount(_MP):
                for _c in os.listdir(_MP):     # rác local — KHÔNG phải Drive
                    _p = os.path.join(_MP, _c)
                    shutil.rmtree(_p, ignore_errors=True) if os.path.isdir(_p) \
                        else os.unlink(_p)
            drive.mount(_MP, force_remount=bool(_try))
        except Exception as _e:  # noqa: BLE001 — thử tiếp vòng sau
            print("   mount lỗi:", _e)
            time.sleep(5)
    if not _drive_alive():
        raise RuntimeError(
            "Không mount được Google Drive sau 4 lần thử — Runtime ▸ "
            "Disconnect and delete runtime rồi Run all lại (tiến độ đã lưu "
            "trên Drive còn nguyên).")

_ensure_drive()
assert Path("/content/drive/MyDrive").exists(), "Drive mount failed — rerun this cell"

PROJECT   = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR
DATA_DIR  = PROJECT / "data"           # organiser dataset (merged packages)
ARTIFACTS = PROJECT / "artifacts"      # everything we build → survives disconnects
# Round-63 (live 07): mkdir NGAY trên mount còn "lười metadata" từng ĐẺ RA một
# AIC2025 SINH ĐÔI rỗng (Drive cho phép trùng tên) — từ đó mỗi phiên mới bind
# ngẫu nhiên vào bản thật hay bản rỗng và "không thấy data". Dự án đã tồn tại
# thì KHÔNG BAO GIỜ mkdir; chỉ khi chờ 3 phút vẫn không thấy (lần setup đầu
# tiên trong đời) mới được tạo.
_t0p = time.time()
while not PROJECT.exists() and time.time() - _t0p < 180:
    print(f"⏳ chưa thấy MyDrive/{DRIVE_PROJECT_DIR} — đợi metadata "
          f"({int(time.time() - _t0p)}s; TUYỆT ĐỐI không tự tạo vội) ...")
    time.sleep(10)
    try:
        list(Path("/content/drive/MyDrive").iterdir())   # cú hích ép nạp metadata
    except OSError:
        pass
if not PROJECT.exists():
    # round-66: KHÔNG BAO GIỜ tự tạo khi chưa được phép — chính là cỗ máy đẻ
    # thư mục dự án sinh đôi. Lần setup đầu tiên THẬT thì bật cờ ở cell 1.
    if not FIRST_TIME_SETUP:
        raise RuntimeError(
            f"3 phút vẫn không thấy MyDrive/{DRIVE_PROJECT_DIR} — máy ảo này "
            "hỏng metadata Drive. Runtime ▸ Disconnect and delete runtime rồi "
            "Run all lại máy mới (dữ liệu trên Drive vẫn nguyên vẹn). Nếu đây "
            "THẬT SỰ là lần đầu tạo dự án: đặt FIRST_TIME_SETUP = True ở cell 1.")
    print(f"⚠ FIRST_TIME_SETUP=True — tạo mới MyDrive/{DRIVE_PROJECT_DIR}.")
# Round-69 (audit): gate chống-sinh-đôi phải phủ cả THƯ MỤC CON — mount thấy
# AIC2025 nhưng chưa nạp children mà mkdir ngay thì data/artifacts sinh đôi
# y hệt vụ round-63, chỉ là một tầng sâu hơn.
for p in (DATA_DIR, ARTIFACTS):
    if p.exists() or FIRST_TIME_SETUP:
        p.mkdir(parents=True, exist_ok=True)
        continue
    _t0c = time.time()
    while not p.exists() and time.time() - _t0c < 120:
        print(f"⏳ chưa thấy {p.name}/ trong dự án — đợi metadata ({int(time.time() - _t0c)}s) ...")
        time.sleep(10)
        try:
            list(PROJECT.iterdir())                  # cú hích ép nạp children
        except OSError:
            pass
    if not p.exists():
        raise RuntimeError(
            f"2 phút không thấy {p.name}/ trong MyDrive/{DRIVE_PROJECT_DIR} — máy "
            "ảo hỏng metadata Drive. Runtime ▸ Disconnect and delete runtime rồi "
            "chạy máy mới; nếu đây là lần setup đầu tiên: FIRST_TIME_SETUP=True.")

# PREFLIGHT (v12): Drive PHẢI ghi/đọc được — quota đầy hay mất quyền thì
# dừng NGAY tại đây thay vì hỏng giữa chừng sau 2 giờ chạy.
_probe = ARTIFACTS / f"_write_test_{int(time.time())}.tmp"
try:
    _probe.write_text("ok", encoding="utf-8")
    assert _probe.read_text(encoding="utf-8") == "ok"
    _probe.unlink()
    print("✅ Drive write test: OK")
except Exception as e:
    raise RuntimeError(
        f"❌ Không ghi được vào Drive ({ARTIFACTS}): {e!r}\n"
        "Kiểm tra dung lượng (quota) Google Drive và quyền truy cập thư mục, "
        "rồi chạy lại ô này."
    ) from e

# DATA-PRESENCE GATE (round-20, live run 9): trên VM mới, DriveFS có thể liệt
# kê data/ ra RỖNG suốt vài phút đầu (metadata sync lười) — mkdir exist_ok ở
# trên còn CHE mất triệu chứng, để cell 7 chết khó hiểu với "Keyframes folder
# not found". Poll tới 3 phút (mỗi listdir là một cú hích ép DriveFS fetch);
# hết kiên nhẫn thì dừng TO với chẩn đoán rõ ràng.
_t0 = time.time()
_data_ok = False
while time.time() - _t0 < 180:
    try:
        if any(DATA_DIR.iterdir()):
            _data_ok = True
            break
    except OSError:
        pass
    print(f"⏳ data/ đang rỗng — đợi DriveFS sync metadata ({int(time.time() - _t0)}s) ...")
    time.sleep(10)
if not _data_ok:
    raise RuntimeError(
        "data/ trên Drive vẫn RỖNG sau 3 phút chờ. Ba nguyên nhân thường gặp:\n"
        "  1) Phiên Colab đăng nhập NHẦM tài khoản Google (kiểm tra avatar góc "
        f"phải trên) — phải là tài khoản có MyDrive/{DRIVE_PROJECT_DIR}/data;\n"
        "  2) DriveFS sync quá chậm — Runtime ▸ Disconnect and delete runtime "
        "rồi Run all lại trên máy mới;\n"
        "  3) Lần chạy đầu tiên mà chưa upload dữ liệu — ném các zip của BTC "
        f"vào MyDrive/{DRIVE_PROJECT_DIR}/data trước (docs/DRIVE_SETUP.md).\n"
        "KHÔNG có gì bị mất — dữ liệu vẫn nằm nguyên trên Drive của tài khoản đúng.")
print(f"✅ data/ nhìn thấy dữ liệu sau {int(time.time() - _t0)}s")

# HF + pip caches on Drive → models/wheels download once, not per session.
os.environ["HF_HOME"] = str(ARTIFACTS / "hf_cache")
os.environ["PIP_CACHE_DIR"] = str(ARTIFACTS / "pip_cache")
for _d in (os.environ["HF_HOME"], os.environ["PIP_CACHE_DIR"]):
    Path(_d).mkdir(parents=True, exist_ok=True)

import shutil
free_gb = shutil.disk_usage(str(PROJECT)).free / 1e9
print(f"Project: {PROJECT}")
print(f"Drive free space: {free_gb:.0f} GB")
if free_gb < 20:
    print("⚠ Less than 20 GB free on Drive — embeddings/checkpoints may not fit!")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3 · Get repo + install dependencies (v12 discipline)            ║
# ╚══════════════════════════════════════════════════════════════════╝
# Quy tắc (học từ notebook Toxicity v12):
#   * check version qua importlib.metadata — KHÔNG import package trước khi
#     nâng cấp (import sớm sẽ ghim version cũ vào sys.modules);
#   * KHÔNG BAO GIỜ đụng torch/torchvision/torchaudio của Colab;
#   * chỉ cài đúng những gói thiếu/sai version (--prefer-binary);
#   * sau khi cài: `pip check` + micro-fix (tối đa 2 vòng, không crash),
#     rồi purge sys.modules TRƯỚC khi import cvp.
FORCE_REINSTALL_DEPS = False

import re, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/content/Core-Vision_Perfect_V1")

def _run(cmd, show=None, **kw):
    # `show` masks credentials in the echoed command — a PAT-carrying clone
    # URL must NEVER be printed into the saved notebook output.
    print("$", " ".join(map(str, show or cmd)))
    return subprocess.run([str(c) for c in cmd], check=False, **kw).returncode

def _pip(args):
    return _run([sys.executable, "-m", "pip", *args])

# Private repo? Add a fine-grained PAT as Colab secret "GITHUB_TOKEN"
# (Contents: Read-only on this repo) and ENABLE its notebook-access toggle.
clone_url, _tok = REPO_URL, None
try:
    from google.colab import userdata
    _tok = userdata.get("GITHUB_TOKEN")
except Exception as _e:
    print(f"⚠ KHÔNG đọc được secret GITHUB_TOKEN ({type(_e).__name__}) — repo "
          "private sẽ KHÔNG clone được. Kiểm tra: 🔑 panel có secret tên đúng "
          "y hệt GITHUB_TOKEN và công tắc 'Notebook access' đã BẬT chưa?")
if _tok and clone_url.startswith("https://github.com/"):
    clone_url = clone_url.replace("https://", f"https://{_tok}@")
    print(f"GITHUB_TOKEN: loaded ({len(_tok)} chars, {_tok[:11]}…)")
elif not _tok:
    print("⚠ GITHUB_TOKEN trống/vắng mặt — thử clone KHÔNG xác thực "
          "(chắc chắn fail nếu repo private).")

if REPO_DIR.exists():
    _run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"])
    _run(["git", "-C", REPO_DIR, "checkout", REPO_REF, "-q"])
    _run(["git", "-C", REPO_DIR, "pull", "-q"])
else:
    rc = _run(["git", "clone", "--branch", REPO_REF, clone_url, REPO_DIR],
              show=["git", "clone", "--branch", REPO_REF, REPO_URL, REPO_DIR])
    if rc != 0:  # private repo / no network → fall back to a Drive copy
        print("⚠ Clone THẤT BẠI. Nguyên nhân thường gặp, theo thứ tự:\n"
              "  1) Secret GITHUB_TOKEN sai tên / chưa bật Notebook access "
              "(xem cảnh báo phía trên);\n"
              "  2) PAT sai/hết hạn/thiếu quyền — cần fine-grained PAT với "
              "Contents: Read-only cấp cho ĐÚNG repo này;\n"
              "  3) Mạng Colab trục trặc tạm thời — chạy lại cell.")
        drive_copy = Path("/content/drive/MyDrive") / DRIVE_PROJECT_DIR / "Core-Vision_Perfect_V1"
        assert drive_copy.exists(), (
            "Clone failed and no Drive copy found. Either make the GitHub repo "
            f"reachable or upload the repo folder to {drive_copy}"
        )
        import shutil as _sh
        _sh.copytree(drive_copy, REPO_DIR)
        print("Using repo copy from Drive")

try:
    from packaging.requirements import Requirement
except ImportError:
    _pip(["install", "-q", "packaging"])
    from packaging.requirements import Requirement
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as _meta_version

# Parse requirements-colab.txt; strip any torch* line (Colab rule #1: the
# preinstalled torch/torchvision/torchaudio build must never be touched).
reqs = []
for _line in (REPO_DIR / "requirements-colab.txt").read_text(encoding="utf-8").splitlines():
    _line = _line.split("#", 1)[0].strip()
    if not _line:
        continue
    try:
        _r = Requirement(_line)
    except Exception:
        print("⚠ bỏ qua requirement không parse được:", _line)
        continue
    if _r.name.lower().replace("-", "_").startswith("torch"):
        print("skip (never touch Colab torch):", _line)
        continue
    reqs.append(_r)

def _satisfied(r):
    """Installed + in range — via importlib.metadata, WITHOUT importing it."""
    try:
        v = _meta_version(r.name)
    except PackageNotFoundError:
        return False
    return (not r.specifier) or r.specifier.contains(v, prereleases=True)

missing = [r for r in reqs if FORCE_REINSTALL_DEPS or not _satisfied(r)]
did_install = bool(missing)
if missing:
    print(f"installing {len(missing)} package(s):", ", ".join(r.name for r in missing))
    _pip(["install", "-q", "--prefer-binary", *[str(r) for r in missing]])
else:
    print("dependencies satisfied — no pip install needed")

_pip(["install", "-q", "-e", str(REPO_DIR), "--no-deps"])

# faiss: gpu wheel with cpu fallback (metadata check — no import)
def _installed(*names):
    for n in names:
        try:
            _meta_version(n)
            return n
        except PackageNotFoundError:
            pass
    return None

if _installed("faiss-gpu-cu12", "faiss-gpu", "faiss-cpu", "faiss") is None:
    if _pip(["install", "-q", "faiss-gpu-cu12"]) != 0:
        _pip(["install", "-q", "faiss-cpu"])
    did_install = True

# `pip check` + micro-fixes for known conflicts (max 2 rounds, then warn)
def _pip_check():
    r = subprocess.run([sys.executable, "-m", "pip", "check"],
                       capture_output=True, text=True)
    return r.returncode, ((r.stdout or "") + "\n" + (r.stderr or "")).strip()

if did_install:
    rc, out = _pip_check()
    for _round in (1, 2):
        if rc == 0:
            break
        # pip's two REAL formats (round-3 fix L-R3-8 — the old regex missed the
        # version-conflict wording so that repair branch never ran):
        #   "pkgA 1.0 requires pkgB, which is not installed."
        #   "pkgA 1.0 has requirement pkgB<2,>=1, but you have pkgB 3.0."
        _specs = sorted({
            m.strip()
            for m in re.findall(
                r"(?:requires|has requirement) (.+?), (?:but you have|which is not installed)", out)
            if not m.strip().lower().startswith("torch")
        })
        if not _specs:
            break
        print(f"pip check micro-fix (round {_round}):", ", ".join(_specs))
        _pip(["install", "-q", "--prefer-binary", *_specs])
        rc, out = _pip_check()
    print("pip check: OK" if rc == 0 else f"⚠ pip check còn cảnh báo (không chặn):\n{out}")

# Purge stale sys.modules of upgraded packages BEFORE importing cvp (v12).
# ONLY the packages actually (re)installed THIS run (round-11): purging every
# requirement dropped numpy/pandas from sys.modules while torch still held
# references to the old modules — the "NumPy module was reloaded" warning.
if did_install:
    _ALIAS = {"pillow": "pil", "pyyaml": "yaml", "opencv_python_headless": "cv2",
              "open_clip_torch": "open_clip", "scikit_learn": "sklearn"}
    _roots = {r.name.lower().replace("-", "_") for r in missing} | {"cvp", "faiss"}
    _roots |= {_ALIAS[n] for n in _roots & set(_ALIAS)}
    _purged = [m for m in list(sys.modules)
               if m.split(".", 1)[0].lower().replace("-", "_") in _roots]
    for _m in _purged:
        sys.modules.pop(_m, None)
    if _purged:
        print(f"purged {len(_purged)} stale sys.modules entries")

    # Sanity (round-11): the HF stack must import cleanly in a FRESH
    # interpreter — a broken hub/accelerate pairing must surface HERE with an
    # actionable message, not 5 cells later as a cryptic circular import.
    _rc = _run([sys.executable, "-c", "import transformers, accelerate"])
    if _rc != 0:
        print("⚠ transformers/accelerate KHÔNG import được — thường do phiên cài "
              "này đã hạ cấp huggingface-hub dưới mức accelerate cần. Cách sửa "
              "sạch nhất: Runtime ▸ Disconnect and delete runtime, rồi Run all "
              "lại từ đầu (mọi tiến độ đã nằm trên Drive, không mất gì).")

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
import cvp
print("cvp", cvp.__version__, "ready")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4 · Point cvp at the data + GPU setup (TF32 / SDPA / bf16)      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, torch

os.environ["CVP_PATHS__DATA_ROOT"]      = str(DATA_DIR)
os.environ["CVP_PATHS__ARTIFACTS_ROOT"] = str(ARTIFACTS)
os.environ["CVP_SETTINGS"] = str(REPO_DIR / "configs" / "settings.yaml")

print("torch", torch.__version__, "| CUDA build", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
GPU_NAME, VRAM_GB, USE_BF16 = "cpu", 0.0, False
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    USE_BF16 = torch.cuda.is_bf16_supported()
    # TF32 fast paths (new API with old fallback)
    try:
        torch.backends.cuda.matmul.fp32_precision = "tf32"
        torch.backends.cudnn.conv.fp32_precision = "tf32"
    except Exception:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    for fn in ("enable_flash_sdp", "enable_mem_efficient_sdp"):
        if hasattr(torch.backends.cuda, fn):
            getattr(torch.backends.cuda, fn)(True)
print(f"GPU: {GPU_NAME} | VRAM {VRAM_GB:.0f} GB | bf16={USE_BF16}")

# Colab secrets → env (optional: Gemini query enhancement/VQA, HF pushes).
# GOOGLE_API_KEY is the name Colab's built-in "Gemini API key ▸ Import from
# Google AI Studio" button creates — the engine accepts either spelling.
try:
    from google.colab import userdata
    # round-90: GEMINI_API_KEY_2/_3 = khóa dự phòng khi khóa chính hết quota ngày
    # round-96: GEMINI_VERTEX_KEY = khóa Vertex express (lane 2 cùng model); HF_TOKEN = lane HF
    for _sec in ("GEMINI_API_KEY", "GOOGLE_API_KEY", "GEMINI_API_KEY_2", "GEMINI_API_KEY_3",
                 "HF_TOKEN", "GEMINI_VERTEX_KEY"):
        try:
            _v = userdata.get(_sec)
            if _v:
                os.environ[_sec] = _v
                print(f"secret {_sec}: loaded")
        except Exception:
            pass
except ImportError:
    pass

from cvp.config import load_settings
from cvp.utils.logging import setup_logging
settings = load_settings()
setup_logging("INFO")
print("data_root      =", settings.paths.data_root)
print("artifacts_root =", settings.paths.artifacts_root)

In [ ]:
# ── 6 · Materialize data → local disk TỪ ZIP GỐC (nhanh + miễn nhiễm FUSE) ──
# Round-14 (live-run 5): copytree 177k JPG lẻ qua Drive FUSE mất 3h+ rồi làm
# SẬP luôn cả mount ([Errno 107] Transport endpoint is not connected — mọi
# file sau đó đọc ra ENOENT). Chiến lược mới: copy CÁC FILE ZIP về local
# (ít file, to, đọc tuần tự — đúng kiểu I/O FUSE làm tốt) rồi giải nén tại
# chỗ — nhanh hơn nhiều lần, resume theo TỪNG zip, tự remount khi FUSE chết.
# Nội dung chỉ-có-trên-Drive (keyframes K-batch tự cắt, csv tái dựng…) được
# merge bù ở pha 2. Embedding/OCR/caption đọc 177k JPG từ local như cũ.
import re as _re
import shutil, time, zipfile
from pathlib import Path

# _ensure_drive/_drive_alive: bản HARDENED định nghĩa ở Ô 2 (round-19) —
# biết gỡ mount chết (fusermount -uz) + dọn mountpoint bẩn trước khi mount lại.
_ensure_drive()          # verify-R14: gate dưới stat qua FUSE — mount phải sống

def _kf_visible() -> bool:
    """DriveFS trên VM mới có thể thấy data/ nhưng CHƯA thấy subdir keyframes/
    (round-28, live nb03 run 5: gate này rơi nhầm sang nhánh Drive-direct rồi
    chết ở catalog). listdir cha = cú hích ép nạp metadata; zip Keyframes*
    cũng được chấp nhận — materialize vốn bung từ zip, không cần dir Drive."""
    try:
        list(DATA_DIR.iterdir())
        if (DATA_DIR / "keyframes").exists():
            return True
        _zn = [p.name.lower().replace("_", "-").replace(" ", "-")
               for p in DATA_DIR.glob("*.zip")]
        return any(n.startswith(("keyframes", "keyframe", "key-frames")) for n in _zn)
    except OSError:
        return False

_kf_ok = False
if COPY_KEYFRAMES_LOCAL:
    for _w in range(30):                       # tới 5 phút (round-62: VM "lười
        if _kf_visible():                      # metadata" từng cần hơn 2 phút)
            _kf_ok = True
            break
        print(f"⏳ DriveFS chưa thấy keyframes/ hay Keyframes*.zip — đợi ({_w * 10}s) ...")
        time.sleep(10)
    if not _kf_ok:
        print("⚠ 5 phút vẫn không thấy keyframes/ lẫn zip nguồn — máy ảo này dính "
              "DriveFS hỏng metadata. KHUYÊN MẠNH: Runtime ▸ Disconnect and "
              "delete runtime rồi Run all lại trên máy mới (dữ liệu Drive vẫn "
              "nguyên). Tạm thời rơi về đọc thẳng Drive (RẤT chậm).")
if COPY_KEYFRAMES_LOCAL and _kf_ok:
    LOCAL_DATA = Path("/content/data")
    LOCAL_DATA.mkdir(exist_ok=True)
    _ZCACHE = Path("/content/__zip_cache")
    _ZCACHE.mkdir(exist_ok=True)
    _VID_DIR_RE = _re.compile(r"^[A-Z]\d{2}_V\d{3}$")

    def _zip_family(zname: str):
        # CÙNG thứ tự ưu tiên với guess_dest ở ô 5 — một zip phải về đúng
        # MỘT family ở cả hai ô. Videos* trả None: video ở lại Drive (symlink).
        z = zname.lower().replace("_", "-").replace(" ", "-")
        if "map" in z and "keyframe" in z:                        return "map-keyframes"
        if "clip-feature" in z or "features-32" in z:             return "clip-features-32"
        if "media-info" in z or "metadata" in z:                  return "media-info"
        if "object" in z:                                         return "objects"
        if z.startswith(("keyframes", "keyframe", "key-frames")): return "keyframes"
        return None

    def _walk_wrapper(root: Path) -> Path:
        # bỏ các folder bọc ngoài thật sự (Keyframes_L26/keyframes/…) nhưng
        # không bao giờ nhầm một payload dir dạng L21_V001 đơn độc là wrapper
        src = root
        while True:
            ch = list(src.iterdir())
            if len(ch) == 1 and ch[0].is_dir() and not _VID_DIR_RE.match(ch[0].name):
                src = ch[0]
                continue
            return src

    def _merge_into(src: Path, dest: Path) -> int:
        """Move src/* vào dest — không ghi đè, đi sâu 1 cấp cho dir trùng."""
        kept = 0
        dest.mkdir(parents=True, exist_ok=True)
        for item in src.iterdir():
            target = dest / item.name
            if not target.exists():
                shutil.move(str(item), str(target))
            elif item.is_dir() and target.is_dir():
                for sub in item.iterdir():
                    st = target / sub.name
                    if not st.exists():
                        shutil.move(str(sub), str(st))
                    else:
                        kept += 1
            else:
                kept += 1
        return kept

    for _sub in ("keyframes", "map-keyframes", "media-info", "objects", "clip-features-32"):
        dst = LOCAL_DATA / _sub
        _stamp = LOCAL_DATA / f".materialized-{_sub}"
        if _stamp.exists():
            # verify-R14: stamp KHÔNG được che zip mới upload giữa session —
            # còn zip matching chưa có marker local thì phải bung bổ sung.
            _ensure_drive()
            _new = [z for z in sorted(DATA_DIR.glob("*.zip"))
                    if _zip_family(z.name) == _sub
                    and not (LOCAL_DATA / f".unzipped-{_sub}-{z.stem}").exists()]
            if not _new:
                print(f"{_sub}: đã materialize trong session này — skip")
                continue
            print(f"{_sub}: {len(_new)} zip mới sau lần materialize trước → bung bổ sung")
        if dst.exists():
            for stale in dst.glob("*.__tmp"):
                shutil.rmtree(stale, ignore_errors=True) if stale.is_dir() else stale.unlink()

        # PHA 1 — bung từ zip nguồn (marker LOCAL theo từng zip → resume mịn;
        # crash giữa merge không sao: lần sau bung lại, merge chỉ bù file thiếu)
        _ensure_drive()
        for zp in sorted(DATA_DIR.glob("*.zip")):
            if _zip_family(zp.name) != _sub:
                continue
            _done = LOCAL_DATA / f".unzipped-{_sub}-{zp.stem}"
            if _done.exists():
                continue
            t0 = time.time()
            lz = _ZCACHE / zp.name
            tmp_root = _ZCACHE / "__tmp_extract"
            for _attempt in (1, 2, 3):
                try:
                    # verify-R14: MỌI syscall chạm FUSE (stat, copyfile) phải
                    # nằm TRONG retry — zip trước mất nhiều phút extract thuần
                    # local, FUSE có thể chết trong cửa sổ đó.
                    _ensure_drive()
                    _free = shutil.disk_usage("/content").free
                    if _free < zp.stat().st_size * 2.2 + 5e9:
                        raise RuntimeError(          # không retry lỗi hết disk
                            f"Disk local sắp đầy ({_free / 1e9:.0f} GB) — không đủ "
                            f"chỗ bung {zp.name}. Runtime ▸ Disconnect and delete "
                            "runtime để lấy máy mới, hoặc đặt "
                            "COPY_KEYFRAMES_LOCAL=False (chậm hơn nhiều).")
                    shutil.copyfile(zp, lz)             # 1 file to, đọc tuần tự
                    if tmp_root.exists():
                        shutil.rmtree(tmp_root)
                    with zipfile.ZipFile(lz) as z:      # CRC check từng member
                        z.extractall(tmp_root)
                    break
                except (OSError, zipfile.BadZipFile) as e:
                    print(f"   ⚠ {zp.name}: {e!r} — thử lại ({_attempt}/3)")
                    if _attempt == 3:
                        raise
                    time.sleep(5)
            kept = _merge_into(_walk_wrapper(tmp_root), dst)
            shutil.rmtree(tmp_root, ignore_errors=True)
            lz.unlink(missing_ok=True)                  # trả disk ngay
            _done.touch()
            print(f"   {zp.name} → local {_sub}/ ({time.time() - t0:.0f}s"
                  + (f", giữ {kept} mục trùng)" if kept else ")"))

        # PHA 2 — merge phần CHỈ có trên Drive (K-batch tự cắt, upload tay…):
        # 1 lần listdir + exists-check local là rẻ; copy lẻ chỉ cho phần thiếu.
        added = 0
        srcD = DATA_DIR / _sub
        # verify-R14: family chỉ-có-folder (không zip nguồn) → pha 1 chưa hề
        # tạo dst; copy2 vào parent chưa tồn tại sẽ FileNotFoundError.
        dst.mkdir(parents=True, exist_ok=True)
        for _attempt in (1, 2, 3):
            try:
                _ensure_drive()                 # srcD.exists cũng chạm FUSE
                if srcD.exists():
                    for item in sorted(srcD.iterdir()):
                        if item.name.startswith(".unzipped-") or item.name.endswith(".__tmp"):
                            continue
                        target = dst / item.name
                        if target.exists():
                            continue
                        tmp_target = dst / (item.name + ".__tmp")
                        if tmp_target.is_dir():
                            shutil.rmtree(tmp_target)
                        elif tmp_target.exists():
                            tmp_target.unlink()
                        (shutil.copytree if item.is_dir() else shutil.copy2)(item, tmp_target)
                        tmp_target.rename(target)
                        added += 1
                break
            except OSError as e:
                print(f"   ⚠ merge Drive-extras {_sub}: {e!r} — thử lại ({_attempt}/3)")
                if _attempt == 3:
                    raise
                time.sleep(5)
        _stamp.touch()
        _n = sum(1 for _ in dst.iterdir())
        print(f"{_sub}: sẵn sàng local ({_n} mục"
              + (f", +{added} bù từ Drive" if added else "") + ")")
    # INTEGRITY + SELF-HEAL (round-11/12, live-run lessons): Google Drive FUSE
    # can serve freshly-written files back EMPTY (buffered writes lost when a
    # session dies mid-sync). Round-11 hit 873 header-less map csvs; round-12
    # hit empty clip-features .npy files that killed the provided_clip32 lane
    # AFTER 8h of GPU work. Validate every LOCAL small-file artifact and heal
    # broken ones straight FROM THE SOURCE ZIP (uploaded long ago = reliably
    # synced), repairing the Drive copy too.
    import zipfile as _zf
    import numpy as _np

    def _bad_csv(f):
        try:
            if f.stat().st_size < 40:
                return True
            with open(f, encoding="utf-8-sig") as fh:
                return sum(1 for _ in fh) < 2          # header only / empty
        except OSError:
            return True

    def _bad_npy(f):
        try:
            if f.stat().st_size < 90:                  # npy header alone is ~64B
                return True
            return _np.load(f, mmap_mode="r").shape[0] == 0
        except Exception:
            return True

    def _bad_empty(f):
        try:
            return f.stat().st_size == 0
        except OSError:
            return True

    # (subdir, glob, zip-name matcher, validator, key depth 1=basename 2=vid/name)
    _HEAL_SPECS = [
        ("map-keyframes", "*.csv",
         lambda z: "map" in z and "keyframe" in z, _bad_csv, 1),
        ("clip-features-32", "*.npy",
         lambda z: "clip-feature" in z or "features-32" in z, _bad_npy, 1),
        ("media-info", "*.json",
         lambda z: "media-info" in z or "metadata" in z, _bad_empty, 1),
        ("objects", "*/*.json",
         lambda z: "object" in z, _bad_empty, 2),
    ]
    for _sub, _pat, _match, _isbad, _depth in _HEAL_SPECS:
        _dirL = LOCAL_DATA / _sub
        if not _dirL.is_dir():
            continue
        _key = (lambda p: p.name) if _depth == 1 else (lambda p: f"{p.parent.name}/{p.name}")
        _bad = [f for f in sorted(_dirL.glob(_pat)) if _isbad(f)]
        if not _bad:
            print(f"{_sub} integrity: OK")
            continue
        print(f"⚠ {len(_bad)} file LOCAL rỗng/hỏng trong {_sub}/ (Drive FUSE mất "
              "dữ liệu?) — tự phục hồi từ zip gốc ...")
        # Zip handles opened ONCE per family (round-13): re-opening a Drive
        # zip per bad file would stall for hours on a family-scale corruption.
        _ensure_drive()                        # verify-R14: ZipFile đọc qua FUSE
        _members, _open_zips = {}, []
        for _z in DATA_DIR.glob("*.zip"):
            _zl = _z.name.lower().replace("_", "-")
            if _match(_zl):
                _zh = _zf.ZipFile(_z)
                _open_zips.append(_zh)
                for _n in _zh.namelist():
                    if not _n.endswith("/"):
                        _parts = Path(_n).parts
                        _members["/".join(_parts[-_depth:])] = (_zh, _n)
        _healed = 0
        for f in _bad:
            _srcz = _members.get(_key(f))
            if not _srcz:
                continue
            _data = _srcz[0].read(_srcz[1])
            if not _data:
                continue
            f.write_bytes(_data)                       # heal LOCAL
            _drv = DATA_DIR / _sub / _key(f)           # heal DRIVE too
            try:
                if not _drv.exists() or _isbad(_drv):
                    _tmpf = _drv.parent / (_drv.name + ".__tmp")
                    _tmpf.write_bytes(_data)
                    _tmpf.replace(_drv)
            except OSError:
                pass
            _healed += 1
        for _zh in _open_zips:
            _zh.close()
        print(f"   phục hồi {_healed}/{len(_bad)}")
        _still = [_key(f) for f in _bad if _isbad(f)]
        if _still:
            raise RuntimeError(
                f"{len(_still)} file trong {_sub}/ vẫn hỏng sau phục hồi "
                f"(vd {_still[:3]}) — kiểm tra zip nguồn còn trong data/ trên "
                "Drive (đừng xóa zip!) rồi chạy lại ô này.")
    # videos stay on Drive (huge); link them in
    _ensure_drive()
    if (DATA_DIR / "videos").exists() and not (LOCAL_DATA / "videos").exists():
        (LOCAL_DATA / "videos").symlink_to(DATA_DIR / "videos")
    import os
    os.environ["CVP_PATHS__DATA_ROOT"] = str(LOCAL_DATA)
    from cvp.config import load_settings
    settings = load_settings()
    print("data_root now:", settings.paths.data_root)
else:
    print("using Drive data_root directly")

In [ ]:
# ── 5b · Compact objects index (một lần, nếu nb01 chưa build) ──
# verify-R23 (HIGH): thiếu objects.parquet thì ObjectBooster rơi về đọc từng
# file json qua Drive FUSE — cộng thêm HÀNG PHÚT mỗi query. Build từ đĩa
# local (ô 5 đã materialize) rồi ghi MỘT file parquet lên Drive artifacts.
from cvp.config import load_settings
from cvp.data.catalog import KeyframeCatalog
from cvp.data.objects_compact import build_objects_index

import time as _t
from pathlib import Path as _P

settings = load_settings()
_pq = settings.paths.art("objects_index") / "objects.parquet"
# round-28: stat lười trên VM mới từng nói parquet "không tồn tại" dù nó nằm
# sẵn trên Drive → suýt rebuild vô ích (và chết nếu data_root chưa local).
# Nudge-poll trước khi kết luận vắng mặt.
for _w in range(6):
    try:
        list(_P(str(settings.paths.artifacts_root)).iterdir())   # nudge metadata
    except OSError:
        pass
    if _pq.exists():
        break
    _t.sleep(5)
if _pq.exists():
    print("objects.parquet: đã có —", _pq)
else:
    catalog = KeyframeCatalog(settings)
    catalog.build()
    print("objects.parquet built:", build_objects_index(settings, catalog))

In [ ]:
# ── 5c · Artifacts đọc-nhiều → đĩa LOCAL (round-26, live nb03 run 2) ──
# Engine mmap embeddings/index từ Drive FUSE → query "lạnh" 114-130s, query
# "ấm" 1.75s. Copy các thư mục CHỈ-ĐỌC về local (~4-6GB, vài phút) rồi trỏ
# artifacts_root vào đó. Drive KHÔNG bị đụng — submissions được đồng bộ ngược
# về Drive ở ô đóng gói.
import os, shutil, time
from pathlib import Path

LOCAL_ART = Path("/content/artifacts")
LOCAL_ART.mkdir(exist_ok=True)
_READ_HOT = ("catalog", "embeddings", "indexes", "text_index", "objects_index",
             "asr", "checkpoints", "thumbs")   # round-42: webp thumbs → lưới UI 10x
_t0 = time.time()
try:
    list(ARTIFACTS.iterdir())    # round-28: nudge metadata trước loạt exists()
except OSError:
    _ensure_drive()
_missing = []
for _d in _READ_HOT:
    src, dst = ARTIFACTS / _d, LOCAL_ART / _d
    if dst.exists():
        print(f"   {_d}/: đã có local — skip")
        continue
    _ensure_drive()
    if not src.exists():
        _missing.append(_d)          # round-76: thiếu là phải LA LÊN, xem dưới
        continue
    _tmp = LOCAL_ART / (_d + ".__tmp")
    if _tmp.exists():
        shutil.rmtree(_tmp)
    shutil.copytree(src, _tmp)
    _tmp.rename(dst)
    print(f"   {_d}/ → local")
if _missing:
    # Round-76 (audit tiền-trận): trước đây thiếu thư mục nào là LẶNG LẼ bỏ
    # qua — text_index vắng mặt nghĩa là OCR/ASR/caption âm thầm = 0 suốt
    # trận. Metadata DriveFS lười là thủ phạm quen mặt; thuốc: đổi máy ảo.
    print(f"\n⚠⚠⚠ THIẾU {len(_missing)} kho artifacts trên Drive: {_missing}")
    print("    Máy ảo lười metadata? ĐỔI MÁY ẢO MỚI rồi Run all lại —")
    print("    KHÔNG ra trận khi thiếu bất kỳ kho nào ngoài 'thumbs'.")
(LOCAL_ART / "submissions").mkdir(parents=True, exist_ok=True)
os.environ["CVP_PATHS__ARTIFACTS_ROOT"] = str(LOCAL_ART)
# round-42: có kho thumbnail (chạy scripts/60_make_thumbs.py MỘT lần) → web
# đội tải ảnh ~8KB thay vì 60-150KB — lưới hiện gần như tức thì qua tunnel.
if (LOCAL_ART / "thumbs").is_dir():
    os.environ["CVP_WEB__THUMBS_DIR"] = str(LOCAL_ART / "thumbs")
    print("thumbs: BẬT (webp 320px)")
print(f"artifacts_root now: {LOCAL_ART} ({time.time() - _t0:.0f}s)")

In [ ]:
# ── L1 · Bench GT từ bài tham chiếu 19.8 ──
RUN_GT = True
import subprocess, sys
def _run(*args):
    # Round-47: stream con-process output VÀO CELL — subprocess.run kế thừa
    # fd thật của kernel nên log 10 giờ ASR từng "im lặng" trong cell (nó chảy
    # vào runtime log, không phải notebook).
    print("$", " ".join(map(str, args)), flush=True)
    p = subprocess.Popen([sys.executable, "-u", *map(str, args)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True)
    for _ln in p.stdout:
        print(_ln, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"lệnh lỗi (exit {p.returncode})")
REF_ZIP   = PROJECT / "queries" / "thunghiem-ref.zip"
TRIAL_DIR = PROJECT / "queries" / "p1"          # 24 đề vòng nháp (đã upload từ trước)
GT_PATH   = PROJECT / "queries" / "gt-thunghiem.json"
if RUN_GT:
    if not REF_ZIP.exists():
        print(f"⚠ Chưa thấy {REF_ZIP} — upload 'submission (3).zip' (bài 19.8) "
              "lên Drive với tên đó rồi chạy lại cell này. Các cell sau vẫn chạy "
              "được nếu GT đã dựng từ trước.")
    else:
        _run(REPO_DIR / "scripts" / "62_build_gt_from_reference.py",
             "--reference", REF_ZIP, "--queries", TRIAL_DIR, "--out", GT_PATH)
print("GT:", "sẵn sàng ✓" if GT_PATH.exists() else "CHƯA có")

In [ ]:
# ── C1 · CHIẾN DỊCH MỘT LẦN (round-88): mọi ý tưởng còn lại, một phiên, một Run all ──
# Mỗi "cánh" = một bench đầy đủ (đúng dàn vũ khí trận, gói ABK) + MỘT thay đổi
# duy nhất → quy được công/tội cho từng ý tưởng. Mỗi cánh xong là lưu riêng
# lên Drive (lab/campaign/<cánh>_run/ trước, rồi <cánh>.json = dấu "đã xong");
# chạy lại notebook thì cánh đã có kết quả được BỎ QUA (xóa .json để đo lại).
# Audit r88: cánh chạy SUY THOÁI (OOM, rớt lane, reranker không build, câu
# không có CSV) KHÔNG được lưu — thà mất một cánh còn hơn một điểm nói dối.
RUN_CAMPAIGN = True
ARMS = "all"   # hoặc danh sách con, vd ["DIVERSE", "ABK+V5"] cho phiên 2
# Round-96: id HF của VLM cục bộ cho hai cánh ABK+LOCALR / ABK+LOCALQA (plan B khi Gemini bão).
# "" = bỏ hai cánh đó. CHỈ điền id đã kiểm chứng tồn tại trên huggingface.co (cell tự kiểm
# trước khi tốn 70 phút); bf16 phải vừa VRAM còn trống (~58 GiB trên A100-80GB).
LOCAL_VLM_ID = "Qwen/Qwen3-VL-8B-Instruct"   # kiểm chứng HF 05/09: 8.8B, Apache-2.0, bf16 16.3 GiB
LOCAL_VLM_ID_2 = "Qwen/Qwen3.5-9B"            # kiểm chứng HF 05/09: VLM gốc, OCRBench 89.2, bf16 18 GiB
                                              # (cần transformers ≥ 5.x có qwen3_5; không có → cánh bỏ)
# Round-96: lane Hugging Face Inference Providers (secret HF_TOKEN; router.huggingface.co, giá của
# provider). Id đã kiểm chứng trên router 05/09/2026; "" hoặc thiếu HF_TOKEN = bỏ hai cánh HF.
HF_ROUTER_QA_MODEL = "Qwen/Qwen3-VL-235B-A22B-Instruct"   # novita $0.30/$1.50, deepinfra
HF_ROUTER_RERANK_MODEL = "zai-org/GLM-5.3-Flash"           # 6 provider, $0.075/$0.25 (KM tới 09/09) rồi $0.15/$0.50
import gc, json, logging, os, random, re, shutil, time, uuid, torch
from pathlib import Path

_camp = PROJECT / "artifacts" / "lab" / "campaign"          # tên Drive khai báo
# Thứ tự (audit r88): cánh chỉ-đổi-retrieval đứng SÁT baseline trong cùng cửa
# sổ quota Gemini; cánh ngốn Gemini (V5) và nặng GPU (DIVERSE) chạy sau.
_ARM_ORDER = ("TUNE", "ABK", "ABK+TUNED", "ABK+W", "ABK+RRF", "DIVERSE", "ABK+V5",
              "MERGE2", "MERGE3", "MERGE_SIB", "MERGE2_NOHEDGE",
              "ABK+BREAKER", "ABK+OCRCTX", "ABK+F6",       # round-96: bài học sơ tuyển 3
              "ABK+G38QA", "ABK+G38R",        # round-89: Gemini 3.8 Flash (GA 02/09/2026)
              "ABK+HFQA", "ABK+HFR",          # round-96: lane Hugging Face (nhà cung cấp độc lập)
              "ABK+LOCALR", "ABK+LOCALQA",    # round-96: VLM cục bộ (nạp thêm ~17 GiB → cuối)
              "ABK+LOCALR2", "ABK+LOCALQA2")  # round-96: VLM cục bộ thứ hai (LOCAL_VLM_ID_2)
_LOCAL_ARM_ID = {"ABK+LOCALR": LOCAL_VLM_ID, "ABK+LOCALQA": LOCAL_VLM_ID,
                 "ABK+LOCALR2": LOCAL_VLM_ID_2, "ABK+LOCALQA2": LOCAL_VLM_ID_2}
_LOCAL_ARMS = tuple(_LOCAL_ARM_ID)
_HF_ARMS = ("ABK+HFQA", "ABK+HFR")
# Round-91: khóa của K chỉ có 250 request/NGÀY cho gemini-3.1-pro-preview (dashboard
# AI Studio 04/09); một cánh bench ≈ 90 cuộc gọi Pro (3 câu QA) → tối đa 2 cánh
# dùng Pro mỗi ngày. G38QA trả lời QA bằng Flash (10.000/ngày) nên đứng trước G38R.
_ARM_NOTE = {
    "TUNE":      "dò lại trọng số fusion trên tower mới → best_weights-candidate.json",
    "ABK":       "baseline trận (đo lại cùng phiên = thước đo nhiễu)",
    "ABK+TUNED": "ABK với trọng số ứng viên vừa dò (shrinkage 50% như trận)",
    "ABK+W":     "trọng số OCR/ASR theo câu hỏi (round-88, heuristic)",
    "ABK+RRF":   "fusion RRF thay weighted_sum (chưa từng đo với dàn hiện tại)",
    "DIVERSE":   "3 lane 45/30/25 + qwen_embed, ĐẦY ĐỦ reranker (lượt nộp 2)",
    "ABK+V5":    "VLM rerank 5 phiếu (tách riêng khỏi gói X)",
    "MERGE2":    "trộn RRF ABK ⊕ DIVERSE (kế hoạch lượt 2 thật, hedge QA)",
    "MERGE3":    "trộn RRF ABK ⊕ DIVERSE ⊕ ABK+V5 (máy thứ 3 có đáng không)",
    "MERGE_SIB": "đối chứng: trộn ABK ⊕ ABK+V5 (cùng đội hình = chỉ nhiễu)",
    "MERGE2_NOHEDGE": "đối chứng: MERGE2 không hedge QA (tách phần thưởng hedge)",
    "ABK+G38R":  "VLM rerank bằng gemini-3.8-flash thay 3.5-flash-lite (round-89)",
    "ABK+G38QA": "QA trả lời bằng gemini-3.8-flash thay 3.1-pro-preview (round-89)",
    "ABK+BREAKER": "cầu dao bão Gemini BẬT — API khỏe thì phải = ABK (round-96)",
    "ABK+OCRCTX": "QA đọc thêm chữ OCR của các khung trong strip, cùng ASR (round-96)",
    "ABK+F6":    "strip QA 6 khung thay 3 (round-96)",
    "ABK+LOCALR": "VLM rerank bằng VLM cục bộ LOCAL_VLM_ID thay Gemini (round-96, plan B)",
    "ABK+LOCALQA": "QA trả lời bằng VLM cục bộ LOCAL_VLM_ID thay Gemini Pro (round-96, plan B)",
    "ABK+LOCALR2": "VLM rerank bằng VLM cục bộ thứ hai LOCAL_VLM_ID_2 (round-96)",
    "ABK+LOCALQA2": "QA trả lời bằng VLM cục bộ thứ hai LOCAL_VLM_ID_2 (round-96)",
    "ABK+HFQA":  "QA trả lời qua HF Inference Providers (HF_ROUTER_QA_MODEL) thay Gemini Pro (round-96)",
    "ABK+HFR":   "VLM rerank qua HF Inference Providers (HF_ROUTER_RERANK_MODEL) thay 3.5-flash-lite (round-96)",
}
_BENCH_ARMS = ("ABK", "ABK+TUNED", "ABK+W", "ABK+RRF", "DIVERSE", "ABK+V5",
               "ABK+G38R", "ABK+G38QA",
               "ABK+BREAKER", "ABK+OCRCTX", "ABK+F6", "ABK+HFQA", "ABK+HFR",
               "ABK+LOCALR", "ABK+LOCALQA", "ABK+LOCALR2", "ABK+LOCALQA2")
_MERGES = {                      # cánh: (các phần, hedge QA)
    "MERGE2": (["ABK", "DIVERSE"], True),
    "MERGE3": (["ABK", "DIVERSE", "ABK+V5"], True),
    "MERGE_SIB": (["ABK", "ABK+V5"], True),
    "MERGE2_NOHEDGE": (["ABK", "DIVERSE"], False),
}
_PACK_ABK = {                                   # = nb03 "Gói knob ABK (round-85)"
    "CVP_SEARCH__NEIGHBOR_CONSISTENCY_BOOST": "0.15",
    "CVP_SEARCH__ROW_STRATEGY": "diversify_tail",
    "CVP_TEMPORAL__SUBMIT_STRATEGY": "jitter",
    "CVP_TEMPORAL__POOL_CONTEXT": "prepend",
    "CVP_TEMPORAL__EVENT_QUERY_VARIANTS": "all",
    "CVP_TEMPORAL__CAPTION_SIGNAL_WEIGHT": "0.2",
    "CVP_VQA__ANSWER_CANONICALIZE": "true",
    "CVP_VQA__ANSWER_NEIGHBOR_FRAMES": "1",
    "CVP_VQA__MAX_CALLS_PER_QUERY": "10",
    "CVP_SEARCH__KIS_MULTI_EVENT": "true",
}
_LINEUP_BATTLE = {"CVP_EMBEDDING__MODEL": "ensemble",
                  "CVP_EMBEDDING__ENSEMBLE_MEMBERS": '["finetuned", "metaclip2"]',
                  "CVP_EMBEDDING__ENSEMBLE_WEIGHTS": "[0.6, 0.4]"}
_LINEUP_DIVERSE = {"CVP_EMBEDDING__MODEL": "ensemble",
                   "CVP_EMBEDDING__ENSEMBLE_MEMBERS": '["finetuned", "metaclip2", "qwen_embed"]',
                   "CVP_EMBEDDING__ENSEMBLE_WEIGHTS": "[0.45, 0.3, 0.25]"}
_ARM_ENV = {
    "ABK": {},
    "ABK+TUNED": {},          # trọng số ứng viên nạp trong _apply_env
    "ABK+W": {"CVP_SEARCH__QUERY_ADAPTIVE_WEIGHTS": "true"},
    "ABK+RRF": {"CVP_SEARCH__FUSION_METHOD": "rrf"},
    "DIVERSE": _LINEUP_DIVERSE,
    "ABK+V5": {"CVP_SEARCH__VLM_RERANK_VOTES": "5"},
    "ABK+G38R": {"CVP_SEARCH__VLM_RERANK_MODEL": "gemini-3.8-flash"},
    "ABK+G38QA": {"CVP_VQA__ANSWER_MODEL": "gemini-3.8-flash"},
    "ABK+BREAKER": {"CVP_BREAKER__ENABLED": "true"},
    "ABK+OCRCTX": {"CVP_VQA__OCR_CONTEXT": "true"},
    "ABK+F6": {"CVP_VQA__FRAMES_PER_ANSWER": "6"},
    "ABK+LOCALR": {"CVP_SEARCH__VLM_RERANK_PROVIDER": "hf_auto",
                   "CVP_VQA__LOCAL_BACKEND": "hf_auto", "CVP_VQA__LOCAL_HF_ID": LOCAL_VLM_ID},
    "ABK+LOCALQA": {"CVP_VQA__PROVIDER": "local",
                    "CVP_VQA__LOCAL_BACKEND": "hf_auto", "CVP_VQA__LOCAL_HF_ID": LOCAL_VLM_ID},
    "ABK+LOCALR2": {"CVP_SEARCH__VLM_RERANK_PROVIDER": "hf_auto",
                    "CVP_VQA__LOCAL_BACKEND": "hf_auto", "CVP_VQA__LOCAL_HF_ID": LOCAL_VLM_ID_2},
    "ABK+LOCALQA2": {"CVP_VQA__PROVIDER": "local",
                     "CVP_VQA__LOCAL_BACKEND": "hf_auto", "CVP_VQA__LOCAL_HF_ID": LOCAL_VLM_ID_2},
    "ABK+HFQA": {"CVP_VQA__ANSWER_MODEL": f"hf:{HF_ROUTER_QA_MODEL}"},
    "ABK+HFR": {"CVP_SEARCH__VLM_RERANK_MODEL": f"hf:{HF_ROUTER_RERANK_MODEL}"},
}
# Mọi khóa env chiến dịch có thể đụng — dọn sạch trước MỖI cánh (env sống
# dai trong kernel; audit r74). Gói D/X cũng dọn phòng khi kernel dùng chung.
_ALL_KEYS = (set(_PACK_ABK) | set(_LINEUP_DIVERSE)
             | {k for e in _ARM_ENV.values() for k in e}
             | {"CVP_SEARCH__HEAD_DIVERSITY", "CVP_SEARCH__QA_MULTI_EVENT",
                "CVP_TEMPORAL__PER_EVENT_TOPK", "CVP_TEMPORAL__MAX_VIDEOS",
                "CVP_VQA__SELF_CONSISTENCY", "CVP_SEARCH__TOPK",
                "CVP_QUERY__EXPANSIONS", "CVP_VQA__PARALLEL_CALLS",
                "CVP_VQA__ANSWER_VARIANT_ROWS", "CVP_VQA__EXACT_TRANSCRIPTION",
                "CVP_SEARCH__VLM_RERANK_VOTES", "CVP_SEARCH__FUSION_METHOD",
                "CVP_SEARCH__QUERY_ADAPTIVE_WEIGHTS",
                "CVP_SEARCH__VLM_RERANK_MODEL", "CVP_VQA__ANSWER_MODEL",
                "CVP_SEARCH__VLM_RERANK_LOCAL_FALLBACK", "CVP_SUBMISSION__QUERY_ORDER",
                "CVP_VQA__OCR_CONTEXT_CHARS"}       # round-96
             | {f"CVP_SEARCH__WEIGHTS__{s}" for s in
                ("VISUAL", "OCR", "ASR", "CAPTION", "METADATA", "OBJECT")})
_BASE_W = {"visual": 1.0, "ocr": 0.35, "asr": 0.30, "caption": 0.25,
           "metadata": 0.15, "object": 0.25}
# Dấu phiên: Δ chỉ tin khi cùng phiên với ABK. Audit r89: giữ nguyên khi chạy
# lại cell trong cùng kernel (chạy lại một cánh hỏng không được đo lại ABK).
SESSION = globals().get("SESSION") or uuid.uuid4().hex[:8]
_GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"


def _shrink(w):
    """Trọng số tuned + shrinkage 50% về mặc định — đúng công thức trận."""
    return {k: round(0.5 * float(w.get(k, v)) + 0.5 * v, 4) for k, v in _BASE_W.items()}


def _read_weights(path):
    """best.weights của một file tune (nudge DriveFS) — None nếu chưa thấy."""
    try:
        list(path.parent.iterdir())
    except OSError:
        pass
    if not path.exists():
        return None
    _w = json.loads(path.read_text(encoding="utf-8")).get("best", {}).get("weights")
    return dict(_w) if _w else None


def _apply_env(arm, battle_w, cand_w=None):
    """Env = bản sao y đội hình trận nb03 (round-87) + thay đổi DUY NHẤT của cánh."""
    for _k in _ALL_KEYS:
        os.environ.pop(_k, None)
    os.environ.update(_LINEUP_BATTLE)
    os.environ["CVP_QUERY__PROVIDER"] = "gemini"
    os.environ["CVP_FINETUNED__CHECKPOINT"] = str(
        Path(os.environ["CVP_PATHS__ARTIFACTS_ROOT"]) / "checkpoints" / "vi_siglip2_best")
    os.environ["CVP_SEARCH__VLM_RERANK"] = "true"
    os.environ["CVP_SEARCH__VLM_RERANK_TOPK"] = "48"
    os.environ["CVP_SEARCH__VLM_RERANK_VOTES"] = "3"
    os.environ["CVP_VQA__SELF_CONSISTENCY"] = "3"
    os.environ["CVP_SEARCH__RERANKER"] = "qwen_reranker"
    os.environ["CVP_SEARCH__LOW_CONFIDENCE_RETRY"] = "true"
    os.environ["CVP_VQA__PARALLEL_CALLS"] = "2"          # = nb03 round-87
    os.environ["CVP_SEARCH__VLM_RERANK_MODEL"] = "gemini-3.5-flash-lite"   # = settings.yaml
    os.environ["CVP_VQA__ANSWER_MODEL"] = "gemini-3.1-pro-preview"        # = settings.yaml
    os.environ.update(_PACK_ABK)
    _w = _shrink(cand_w) if arm == "ABK+TUNED" else dict(battle_w)
    for _sig, _val in _w.items():
        os.environ[f"CVP_SEARCH__WEIGHTS__{_sig.upper()}"] = str(_val)
    os.environ.update(_ARM_ENV.get(arm, {}))
    return _w


class _StormCounter(logging.Handler):
    """Đếm theo cánh: bão API (môi trường) và SUY THOÁI cấu trúc (audit r88:
    mọi đường hỏng trong engine — OOM, rớt lane, reranker chết — chỉ là một
    dòng WARNING bị nuốt; phải bắt lại thành số và chặn lưu)."""
    STORM = re.compile(r"\b(429|503|504)\b|RESOURCE_EXHAUSTED|UNAVAILABLE|DEADLINE")
    # Audit r88b: "keeping original order" cũng là log khi VLM rerank (Gemini)
    # bị bão → cross-encoder chỉ nhận dạng bằng cụm riêng của nó; lỗi VLM là
    # môi trường (SOFT): đếm, in, và chỉ chặn khi rớt trên >1/4 số câu.
    FATAL = ("out of memory", "OutOfMemory", "failed at query time",
             "CONTINUING without", "EVERY dense lane failed",
             "Cross-encoder rerank failed", "Cross-encoder returned",
             "failed to build", "DISABLED until", "SKIPPING this lane")
    SOFT = ("no usable scores", "VLM rerank failed", "VLM vote", "trying local model",
            "Local VQA failed", "Loading local VQA", "Low-confidence retry failed",
            "cầu dao bão")                      # round-96: cầu dao mở/đóng = môi trường
    # Round-96: VLM cục bộ nạp (INFO của cvp.models.local_vlm) — cánh LOCAL* phải thấy dòng này
    LOCAL_LOAD = "Loading local hf_auto VLM"
    # Audit r89: chuỗi dự phòng rớt model (3.8 → 3.7) chỉ là một WARNING —
    # đếm theo model để cánh "đo model X" không âm thầm đo model Y. Chỉ khớp
    # dạng THƯỜNG (không có "(economical)" = retry cùng model, chưa rớt).
    FALLBACK = re.compile(r"Gemini model '([^']+)' failed \(.*— trying next")
    # Round-92: đếm 200/lỗi THEO MODEL từ log httpx (INFO) — cánh "đo model X"
    # được chấm theo tỷ lệ X TỰ trả lời, không phải theo số lần rớt tuyệt đối.
    HTTP = re.compile(r"models/([^:/]+):generateContent \"HTTP/1\.1 (\d{3})")

    def __init__(self):
        super().__init__(level=logging.INFO)     # INFO chỉ để hứng httpx
        self.reset()

    def emit(self, record):
        try:
            _m = record.getMessage()
        except Exception:      # noqa: BLE001 — bộ đếm không được làm hỏng log
            return
        if record.name == "httpx":
            _hm = self.HTTP.search(_m)
            if _hm:
                _d = self.http.setdefault(_hm.group(1), {})
                _d[_hm.group(2)] = _d.get(_hm.group(2), 0) + 1
            return
        if record.name.startswith("cvp") and self.LOCAL_LOAD in _m:
            self.local_loads += 1                # round-96 (INFO)
        if not record.name.startswith("cvp") or record.levelno < logging.WARNING:
            return
        for _mt in self.STORM.finditer(_m):      # group(1) = mã số, else từ khóa
            _k = _mt.group(1) or _mt.group(0)
            self.storm[_k] = self.storm.get(_k, 0) + 1
        for _d in self.FATAL:
            if _d.lower() in _m.lower():
                self.fatal[_d] = self.fatal.get(_d, 0) + 1
                break
        for _d in self.SOFT:
            if _d.lower() in _m.lower():
                self.degraded[_d] = self.degraded.get(_d, 0) + 1
                break
        if record.levelno >= logging.ERROR:
            self.errors += 1
        _fb = self.FALLBACK.search(_m)
        if _fb:
            self.fallback[_fb.group(1)] = self.fallback.get(_fb.group(1), 0) + 1
        if "exceeded your current quota" in _m.lower():   # round-91: hết quota NGÀY
            self.quota += 1

    def reset(self):
        self.storm, self.fatal, self.degraded, self.errors = {}, {}, {}, 0
        self.fallback = {}
        self.quota = 0
        self.http = {}
        self.local_loads = 0


if RUN_CAMPAIGN and not GT_PATH.exists():
    print("⚠ Chưa có GT — chạy cell L1 trước.")
if RUN_CAMPAIGN and GT_PATH.exists():
    from cvp.config import load_settings
    from cvp.eval.official import score_run
    from cvp.pipeline.attempts import load_run, rrf_merge_runs, write_merged
    from cvp.pipeline.auto_agent import run_auto
    from cvp.search import cross_rerank as _cr
    from cvp.search.engine import SearchEngine
    from cvp.search.query_cues import query_signal_cues
    from cvp.utils.io import atomic_write_json

    # ── Tiền kiểm (audit r88): mọi thứ có thể làm 9 giờ bench đo SAI đội hình
    #    phải chặn NGAY BÂY GIỜ, không phải sau khi tốn một đêm.
    assert os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"), (
        "Chiến dịch CẦN Gemini (enhancement + VLM rerank + QA Pro) — không có key thì "
        "mọi cánh đo cấu hình KHÁC trận. Bật 'Notebook access' cho secret "
        "GEMINI_API_KEY, chạy lại cell 4 rồi cell này.")
    _qfiles = sorted(Path(TRIAL_DIR).glob("query-*.txt"))
    assert _qfiles, f"TRIAL_DIR trống: {TRIAL_DIR}"
    _BATTLE_RAW = None
    for _try in range(6):                        # nudge-poll như NB3_OBJECTS
        _BATTLE_RAW = _read_weights(PROJECT / "artifacts" / "tuning" / "best_weights.json")
        if _BATTLE_RAW:
            break
        time.sleep(5)
    if _BATTLE_RAW is None:
        raise RuntimeError("KHÔNG đọc được artifacts/tuning/best_weights.json — bench không "
                           "phải bản sao trận thì KHÔNG chạy 9 giờ. Đổi máy ảo mới rồi Run all.")
    _BATTLE_W = _shrink(_BATTLE_RAW)             # đọc MỘT lần, dùng cho mọi cánh
    print("⚖ trọng số trận (tuned + shrink 50%):", _BATTLE_W)
    for _p in (PROJECT / "artifacts" / "lab", _camp):    # chống thư mục sinh đôi
        try:
            list(_p.parent.iterdir())
        except OSError:
            pass
        for _ in range(6):
            if _p.exists():
                break
            time.sleep(5)
        _p.mkdir(parents=True, exist_ok=True)
    _local = Path(os.environ["CVP_PATHS__ARTIFACTS_ROOT"])
    _arms = list(_ARM_ORDER) if ARMS == "all" else [a for a in _ARM_ORDER if a in ARMS]
    assert _arms, f"ARMS lạ: {ARMS!r} — chọn trong {_ARM_ORDER}"
    _skip_local = [a for a in _arms if a in _LOCAL_ARMS and not _LOCAL_ARM_ID[a]]
    if _skip_local:
        # round-96: không đoán id — không có id đã kiểm chứng thì cánh cục bộ chờ phiên sau
        print(f"⏭ LOCAL_VLM_ID(_2) trống — bỏ {_skip_local} (điền id HF đã kiểm chứng ở cell C1 rồi "
              "chạy lại: chỉ các cánh đó được đo thêm)")
        _arms = [a for a in _arms if a not in _skip_local]
    _hf_ok = bool(os.environ.get("HF_TOKEN")) and bool(HF_ROUTER_QA_MODEL) and bool(HF_ROUTER_RERANK_MODEL)
    if not _hf_ok and any(a in _arms for a in _HF_ARMS):
        print(f"⏭ thiếu secret HF_TOKEN hoặc HF_ROUTER_*_MODEL trống — bỏ {[a for a in _arms if a in _HF_ARMS]}")
        _arms = [a for a in _arms if a not in _HF_ARMS]
    _storm = _StormCounter()
    logging.getLogger().addHandler(_storm)
    _results = {}

    def _now():
        return time.strftime("%Y-%m-%dT%H:%M:%S")

    def _write_json(path, obj):
        # atomic_write_json, dự phòng ghi thường — DriveFS đôi khi từ chối rename
        try:
            atomic_write_json(path, obj)
        except OSError as _e:
            print(f"   ⚠ ghi nguyên tử thất bại ({_e}) — ghi thường: {Path(path).name}")
            Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2),
                                  encoding="utf-8")

    def _done(arm):
        try:
            list(_camp.iterdir())
        except OSError:
            pass
        _p = _camp / f"{arm}.json"
        if not _p.exists():
            return None
        try:
            _d = json.loads(_p.read_text(encoding="utf-8"))
        except (OSError, ValueError) as _e:
            print(f"   ⚠ {arm}.json hỏng ({_e}) — đo lại")
            return None
        if arm == "TUNE" and not ("cv" in _d and "report" in _d):
            print("   ⚠ TUNE.json đời cũ (thiếu cv/report) — dò lại")
            return None
        if "per_query" in _d:                    # dấu xong phải đi kèm đủ CSV
            try:
                list((_camp / f"{arm}_run").iterdir())   # nudge DriveFS
            except OSError:
                pass
            _stems = {p.stem for p in (_camp / f"{arm}_run").glob("query-*.csv")}
            if not set(_d["per_query"]) <= _stems:
                print(f"   ⚠ {arm}: có .json nhưng {arm}_run/ thiếu CSV — đo lại")
                return None
        return _d

    def _save(arm, payload, run_dir=None):
        """CSV lên Drive TRƯỚC (chép ra __tmp, kiểm đủ, đổi tên), .json SAU
        và nguyên tử = dấu xong. Chết giữa chừng thì không có dấu → đo lại."""
        if run_dir is not None:
            # Audit r88b: không phụ thuộc vào xóa trên DriveFS — tmp mang dấu phiên
            # (không đụng tmp của phiên chết), thư mục cũ được XOAY sang -prev bằng
            # rename (một thao tác metadata) thay vì xóa đệ quy.
            _dst, _tmp = _camp / f"{arm}_run", _camp / f"{arm}_run.__tmp-{SESSION}"
            if _tmp.exists():
                shutil.rmtree(_tmp)
            shutil.copytree(run_dir, _tmp)
            _want = {p.name for p in Path(run_dir).glob("query-*.csv")}
            _got = set()
            for _ in range(3):
                try:
                    list(_tmp.iterdir())
                except OSError:
                    pass
                _got = {p.name for p in _tmp.glob("query-*.csv")}
                if _got == _want:
                    break
                time.sleep(2)
            assert _got == _want, (f"[{arm}] chép lên Drive THIẾU {sorted(_want - _got)} "
                                   "— không ghi dấu xong")
            try:
                list(_camp.iterdir())
            except OSError:
                pass
            if _dst.exists():
                _prev = _camp / f"{arm}_run-prev"
                _i = 2
                while _prev.exists():
                    _prev = _camp / f"{arm}_run-prev{_i}"
                    _i += 1
                _dst.rename(_prev)
                print(f"   ↪ {arm}_run/ cũ xoay sang {_prev.name}/")
            try:
                _tmp.rename(_dst)
            except OSError as _e:                # DriveFS từ chối rename → chép + kiểm lại
                print(f"   ⚠ rename thất bại ({_e}) — chép thường")
                shutil.copytree(_tmp, _dst, dirs_exist_ok=True)
                assert {q.name for q in _dst.glob("query-*.csv")} == _want, \
                    f"[{arm}] chép sang {_dst.name} THIẾU CSV — không ghi dấu xong"
                shutil.rmtree(_tmp, ignore_errors=True)
        if arm == "ABK":                         # audit r89: xoay baseline phiên khác
            _abk_p = _camp / "ABK.json"          # NGAY TRƯỚC khi ghi bản mới
            if _abk_p.exists():
                try:
                    _old = json.loads(_abk_p.read_text(encoding="utf-8"))
                except (OSError, ValueError):
                    _old = {}                    # hỏng cũng xoay đi, không đè
                if _old.get("session") != SESSION:
                    _k, _rot = 2, _camp / "ABK-prev.json"
                    while _rot.exists():
                        _rot = _camp / f"ABK-prev{_k}.json"
                        _k += 1
                    try:
                        _abk_p.rename(_rot)
                    except OSError as _e:        # DriveFS từ chối rename → chép + xóa
                        print(f"   ⚠ rename ABK.json thất bại ({_e}) — chép thường")
                        shutil.copy2(_abk_p, _rot)
                        _abk_p.unlink()
                    print(f"   ↪ ABK phiên {_old.get('session')} → {_rot.name} (thêm một lần đo nhiễu)")
        _write_json(_camp / f"{arm}.json", payload)
        _results[arm] = payload

    def _bench(arm):
        _cand = None
        if arm == "ABK+TUNED":
            _cand = _read_weights(_camp / "best_weights-candidate.json")
            if _cand is None:
                raise RuntimeError("ABK+TUNED cần lab/campaign/best_weights-candidate.json "
                                   "— cánh TUNE chưa chạy hoặc hỏng")
            _tdelta = (_results.get("TUNE") or {}).get("report", {}).get("delta")
            if _tdelta is not None and _tdelta <= 0:
                _save(arm, {"arm": arm, "note": _ARM_NOTE[arm], "skipped": True,
                            "reason": f"tuner không thắng in-sample (Δ {_tdelta:+.4f}) — "
                                      "ứng viên = mặc định, không phải trọng số dò được",
                            "session": SESSION})
                print("   ⏭ tuner không thắng in-sample — bỏ ABK+TUNED")
                return _results[arm]
            _diff = max(abs(_shrink(_cand)[k] - _BATTLE_W[k]) for k in _BASE_W)
            if _diff < 0.01:                     # audit r88: không đo cái giống hệt
                _save(arm, {"arm": arm, "note": _ARM_NOTE[arm], "skipped": True,
                            "reason": f"ứng viên ≡ trọng số trận (lệch tối đa {_diff:.3f})",
                            "abk_weights": _BATTLE_W, "candidate_weights": _shrink(_cand),
                            "session": SESSION})
                print(f"   ⏭ ứng viên ≡ trận (lệch {_diff:.3f}) — bỏ, không tốn 70 phút")
                return _results[arm]
        _w = _apply_env(arm, _BATTLE_W, _cand)
        settings = load_settings()               # cũng cấu hình cầu dao bão theo cánh (r96)
        from cvp.models.gemini_health import HEALTH as _HEALTH
        _HEALTH.reset()                          # sổ cầu dao sạch cho mỗi cánh
        _want = (["finetuned", "metaclip2", "qwen_embed"] if arm == "DIVERSE"
                 else ["finetuned", "metaclip2"])
        # Thay đổi duy nhất phải ĂN vào settings, nền phải đúng trận (audit r88)
        _took = {"ABK+W": settings.search.query_adaptive_weights is True,
                 "ABK+RRF": settings.search.fusion_method == "rrf",
                 "ABK+V5": settings.search.vlm_rerank_votes == 5,
                 "DIVERSE": list(settings.embedding.ensemble_members) == _want,
                 "ABK+G38R": settings.search.vlm_rerank_model == "gemini-3.8-flash",
                 "ABK+G38QA": settings.vqa.answer_model == "gemini-3.8-flash",
                 "ABK+BREAKER": settings.breaker.enabled is True and _HEALTH.enabled is True,
                 "ABK+OCRCTX": settings.vqa.ocr_context is True,
                 "ABK+F6": settings.vqa.frames_per_answer == 6,
                 "ABK+HFQA": settings.vqa.answer_model == f"hf:{HF_ROUTER_QA_MODEL}",
                 "ABK+HFR": settings.search.vlm_rerank_model == f"hf:{HF_ROUTER_RERANK_MODEL}"}
        if arm in _LOCAL_ARMS:
            _took[arm] = (settings.vqa.local_backend == "hf_auto"
                          and settings.vqa.local_hf_id == _LOCAL_ARM_ID[arm] != ""
                          and (settings.search.vlm_rerank_provider == "hf_auto" if arm.startswith("ABK+LOCALR")
                               else settings.vqa.provider == "local"))
        assert _took.get(arm, True), f"[{arm}] knob KHÔNG ăn vào settings — sai tên env"
        if arm not in ("ABK+BREAKER",):          # nền: cầu dao TẮT ở mọi cánh khác (= trận cũ)
            assert settings.breaker.enabled is False and _HEALTH.enabled is False, \
                f"[{arm}] cầu dao bão đang BẬT ngoài cánh ABK+BREAKER — env rò"
        if arm in _LOCAL_ARMS:
            # Round-96 tiền kiểm: id phải TỒN TẠI trên HF trước khi tốn 70 phút (bão mạng → thử lại)
            from huggingface_hub import model_info as _hf_info
            _lid = _LOCAL_ARM_ID[arm]
            for _try in range(3):
                try:
                    _hf_info(_lid)
                    break
                except Exception as _e:   # noqa: BLE001 — phân loại rồi quyết
                    if "404" in str(_e) or "Repository Not Found" in str(_e) or _try == 2:
                        raise RuntimeError(f"[{arm}] id {_lid!r} không tra được trên "
                                           f"huggingface.co ({type(_e).__name__}: {str(_e)[:120]}) — "
                                           "không bench 70 phút với id sai") from _e
                    time.sleep(15)
            print(f"   ✓ {_lid} có trên HF")
        assert (settings.search.reranker == "qwen_reranker" and settings.search.vlm_rerank
                and settings.search.vlm_rerank_topk == 48 and settings.vqa.parallel_calls == 2
                and settings.search.kis_multi_event and settings.search.low_confidence_retry
                and abs(settings.search.weights.ocr - _w["ocr"]) < 1e-6), \
            f"[{arm}] env nền KHÔNG đúng trận"
        # Audit r89: cánh "đo model X" phải chứng minh X gọi được TRƯỚC 70 phút
        # (gọi thẳng, không qua chuỗi dự phòng để không bị che), và sau khi chạy
        # không được rớt về model khác quá 1/4 số câu.
        _mid = next((v for k, v in _ARM_ENV.get(arm, {}).items() if k.endswith("_MODEL")), None)
        if _mid:
            from cvp.models.query_processor import _call_with_timeout, economical_config
            from cvp.search.vqa import make_gemini_client
            _cfg = economical_config(_mid) if arm == "ABK+G38R" else None
            _cl = make_gemini_client(settings)
            # Round-92: 503/429/504/timeout là bão TẠM (phiên 90485020: một 503 duy
            # nhất đã giết cả hai cánh 3.8 trong 3 giây) → thử lại 6 lần cách 20 s;
            # 4xx khác (404/400/403) = model chết với khóa này → dừng ngay.
            for _try in range(6):
                try:
                    _call_with_timeout(
                        lambda: _cl.models.generate_content(
                            model=_mid, contents="ping",
                            **({"config": _cfg} if _cfg is not None else {})), 45.0)
                    break
                except Exception as _e:   # noqa: BLE001 — phân loại rồi quyết
                    _msg = str(_e)
                    _transient = any(t in _msg for t in (
                        "503", "429", "504", "UNAVAILABLE", "RESOURCE_EXHAUSTED",
                        "DEADLINE", "wall clock", "timed out"))
                    if not _transient or _try == 5:
                        raise RuntimeError(
                            f"[{arm}] {_mid} KHÔNG gọi được sau {_try + 1} lần "
                            f"({type(_e).__name__}: {_msg[:160]}) — không bench 70 phút với "
                            "model chết") from _e
                    print(f"   ⏳ {_mid} bão ({_msg[:70]}…) — thử lại {_try + 2}/6 sau 20 s")
                    time.sleep(20)
            print(f"   ✓ {_mid} trả lời tiền kiểm")
        _storm.reset()
        # Round-90 (phiên 850660ee): lane metaclip2 rớt vì "[Errno 5] Input/output
        # error" đọc HF cache trên Drive — lỗi tạm thời của DriveFS. Nudge + thử
        # nạp lại tối đa 3 lần TRƯỚC khi bỏ cánh (loader giờ tự tải lại về đĩa
        # cục bộ khi cache Drive hỏng).
        for _try in range(3):
            engine = SearchEngine(settings)
            if engine.member_names == _want or _try == 2:
                break
            print(f"   ⚠ lần {_try + 1}: ensemble thiếu lane {engine.member_names} ≠ {_want} "
                  "— nudge Drive, thử lại sau 20s")
            for _d in ("embeddings", "indexes", "checkpoints", "hf_cache"):
                try:
                    list((PROJECT / "artifacts" / _d).iterdir())
                except OSError:
                    pass
            del engine
            gc.collect()
            torch.cuda.empty_cache()
            time.sleep(20)
        # Round-71: bench thiếu lane là đo SAI đội hình — chặn TRƯỚC khi tốn giờ.
        assert engine.member_names == _want, (
            f"[{arm}] ensemble thiếu lane: {engine.member_names} ≠ {_want} — "
            "index/checkpoint chưa nạp đủ (DriveFS lười?). Đổi máy ảo chạy lại; "
            "KHÔNG bench thiếu lane.")
        # Audit r88: nạp HẾT đội hình lên GPU ngay phút 1 (lane qwen_embed 8B
        # nạp lười; cross-reranker 8B là singleton, build hỏng thì CHỐT False
        # cho mọi cánh sau) rồi mới đo VRAM — DIVERSE có chạy nổi 40GB hay
        # không phải biết ở đây, không phải sau 70 phút.
        engine.search_prepared("a man walking on a street", skip_rerank=True)
        if _cr._RERANKER is False:               # chỉ xóa chốt HỎNG, giữ singleton tốt
            _cr._RERANKER, _cr._RERANKER_KEY = None, None
        assert _cr._get_reranker(settings) is not None, (
            f"[{arm}] cross-reranker KHÔNG build được — không bench thiếu reranker")
        _free, _total = torch.cuda.mem_get_info()
        print(f"   ✓ lane {engine.member_names} · reranker OK · VRAM trống "
              f"{_free / 2**30:.1f}/{_total / 2**30:.1f} GiB · trọng số {_w}")
        if _free < 4 * 2**30:
            raise RuntimeError(f"[{arm}] VRAM trống {_free / 2**30:.1f} GiB < 4 — đội hình "
                               "này KHÔNG chạy nổi trên máy này (câu trả lời cho lượt 2)")
        torch.cuda.reset_peak_memory_stats()
        if _storm.fatal:
            raise RuntimeError(f"[{arm}] suy thoái ngay khi nạp: {_storm.fatal}")
        _hits = None
        if arm == "ABK+W":                       # đúng như run_query_file: bỏ TRAKE, mô tả + câu hỏi
            from cvp.pipeline.run_queries import load_query_lines, parse_query_lines
            from cvp.submission.packager import infer_task
            _hits = []
            for _qp in _qfiles:
                _task = infer_task(_qp.name)
                if _task == "trake":
                    continue
                _rt, _qq = parse_query_lines(_task, load_query_lines(_qp))
                if query_signal_cues(" ".join(t for t in (_rt, _qq) if t)):
                    _hits.append(_qp.stem)
        if _hits is not None:
            print(f"   🎯 cue bắn vào {len(_hits)}/{len(_qfiles)} câu: {_hits}")
        _out = _local / "submissions" / f"camp_{arm.replace('+', '_')}"
        shutil.rmtree(_out, ignore_errors=True)
        _t0, _start = time.time(), _now()
        rep = run_auto(TRIAL_DIR, _out, settings, submit=False, engine_factory=lambda _s: engine)
        if rep.failed:
            raise RuntimeError(f"[{arm}] {len(rep.failed)} câu KHÔNG có CSV: "
                               f"{sorted(rep.failed)} — không lưu")
        if _storm.fatal:
            raise RuntimeError(f"[{arm}] chạy SUY THOÁI (không phải đội hình đã khai): "
                               f"{_storm.fatal} — không lưu")
        _vlm_off = _storm.degraded.get("no usable scores", 0)
        if _vlm_off > len(_qfiles) // 4:
            raise RuntimeError(f"[{arm}] VLM rerank rớt trên {_vlm_off}/{len(_qfiles)} câu (bão "
                               "Gemini kéo dài) — không phải đội hình trận, không lưu")
        if arm in _LOCAL_ARMS:
            # Round-96: cánh cục bộ phải THẬT SỰ nạp model cục bộ, và cánh LOCALQA không được
            # để Gemini Pro trả lời (điểm phải thuộc về model đã khai).
            if _storm.local_loads < 1:
                raise RuntimeError(f"[{arm}] VLM cục bộ {_LOCAL_ARM_ID[arm]} KHÔNG được nạp trong lượt "
                                   "chạy — điểm không thuộc về model đã khai, không lưu")
            _pro_calls = sum(_storm.http.get("gemini-3.1-pro-preview", {}).values())
            if arm.startswith("ABK+LOCALQA") and _pro_calls:
                raise RuntimeError(f"[{arm}] Gemini Pro vẫn được gọi {_pro_calls} lần — QA không "
                                   "hoàn toàn do model cục bộ trả lời, không lưu")
        _fb = _storm.fallback.get(_mid, 0) if _mid else 0
        _share = None
        if _mid:
            _h = _storm.http.get(_mid, {})
            _ok = _h.get("200", 0)
            _bad = sum(v for k, v in _h.items() if k != "200")
            if _ok + _bad:                        # round-92: tỷ lệ model đã khai TỰ trả lời
                _share = _ok / (_ok + _bad)
                if _share < 0.75:
                    raise RuntimeError(f"[{arm}] {_mid} chỉ tự trả lời {_share:.0%} cuộc gọi "
                                       f"({_ok} OK / {_bad} hỏng) — điểm không thuộc về {_mid}, "
                                       "không lưu")
            elif _fb > len(_qfiles) // 4:         # không có log httpx → luật cũ
                raise RuntimeError(f"[{arm}] {_mid} rớt về model khác {_fb} lần (> 1/4 số câu) — "
                                   f"điểm không thuộc về {_mid}, không lưu")
        r = score_run(_out, GT_PATH)
        payload = {**r.to_dict(), "arm": arm, "note": _ARM_NOTE[arm],
                   "env": {k: os.environ[k] for k in sorted(_ALL_KEYS) if k in os.environ},
                   "weights": _w, "minutes": round((time.time() - _t0) / 60, 1),
                   "started_at": _start, "ended_at": _now(), "session": SESSION, "gpu": _GPU,
                   "storm": dict(_storm.storm), "degraded": dict(_storm.degraded),
                   "model_fallbacks": dict(_storm.fallback), "declared_model": _mid,
                   "declared_share": None if _share is None else round(_share, 3),
                   "http_by_model": {m: dict(c) for m, c in _storm.http.items()},
                   "quota_429": _storm.quota,
                   "local_loads": _storm.local_loads, "breaker": _HEALTH.snapshot(),
                   "log_errors": _storm.errors,
                   "vram_free_start_gib": round(_free / 2**30, 1),
                   "vram_peak_gib": round(torch.cuda.max_memory_allocated() / 2**30, 1)}
        if _hits is not None:
            payload["cue_hits"] = _hits
        _save(arm, payload, _out)
        return payload

    def _tune():
        _apply_env("ABK", _BATTLE_W)
        # Dump tín hiệu THUẦN (audit r74): không knob, không reranker, không
        # VLM — dump chụp tín hiệu TRƯỚC rerank, chạy reranker chỉ đốt quota.
        # Audit r88: tuner xuất phát từ MẶC ĐỊNH (giao thức round-44 mà công
        # thức shrink 50% giả định), không từ trọng số đã shrink của trận.
        for _k in list(_PACK_ABK) + ["CVP_SEARCH__QUERY_ADAPTIVE_WEIGHTS"]:
            os.environ.pop(_k, None)
        for _s in _BASE_W:
            os.environ.pop(f"CVP_SEARCH__WEIGHTS__{_s.upper()}", None)
        os.environ["CVP_SEARCH__RERANKER"] = "none"
        os.environ["CVP_SEARCH__VLM_RERANK"] = "false"
        os.environ["CVP_SEARCH__LOW_CONFIDENCE_RETRY"] = "false"
        _dump = _local / "signal_dumps" / "campaign"
        _cand = _camp / "best_weights-candidate.json"      # KHÔNG đụng file trận
        _t0, _start = time.time(), _now()
        _run(REPO_DIR / "scripts" / "23_dump_signals.py",
             "--query-dir", TRIAL_DIR, "--out-dir", _dump)
        _run(REPO_DIR / "scripts" / "21_tune_weights.py", "--signals-dir", _dump,
             "--gt", GT_PATH, "--method", "random", "--trials", "400", "--out", _cand)
        _rep = json.loads(_cand.read_text(encoding="utf-8"))
        # Audit r88: dò và chấm trên CÙNG 23 câu = overfit. Ước lượng held-out
        # bằng 2-fold lặp 8 lần (16 fold, vài giây CPU): dò trên nửa này, chấm
        # nửa kia, so ứng viên (đã shrink) với trọng số trận và mặc định.
        import importlib.util
        _spec = importlib.util.spec_from_file_location(
            "tune21", REPO_DIR / "scripts" / "21_tune_weights.py")
        _m = importlib.util.module_from_spec(_spec)
        _spec.loader.exec_module(_m)
        _sig = _m.load_signals(_dump)
        _gt = {str(k): v for k, v in json.loads(Path(GT_PATH).read_text(encoding="utf-8")).items()}
        _scorer, _sname = _m.resolve_scorer()
        _stems = sorted(s for s in _sig if s in _gt)
        _cv = {"candidate": [], "battle_insample": [], "default": []}
        _net = 0
        for _seed in range(8):
            _sh = list(_stems)
            random.Random(_seed).shuffle(_sh)
            _half = len(_sh) // 2
            for _train, _test in ((_sh[:_half], _sh[_half:]), (_sh[_half:], _sh[:_half])):
                _r = _m.tune({s: _sig[s] for s in _train}, {s: _gt[s] for s in _train},
                             dict(_BASE_W), trials=200, seed=_seed,
                             scorer=_scorer, scorer_name=_sname)
                _norm = _m.normalize_signals({s: _sig[s] for s in _test})
                _gt_t = {s: _gt[s] for s in _test}
                _c = _m.evaluate_weights(_shrink(_r["best"]["weights"]), _norm, _gt_t, _scorer)[0]
                _b = _m.evaluate_weights(dict(_BATTLE_W), _norm, _gt_t, _scorer)[0]
                _d = _m.evaluate_weights(dict(_BASE_W), _norm, _gt_t, _scorer)[0]
                _cv["candidate"].append(_c)
                _cv["battle_insample"].append(_b)
                _cv["default"].append(_d)
                _net += 1 if _c > _d else (-1 if _c < _d else 0)
        _cv_mean = {k: round(sum(v) / len(v), 4) for k, v in _cv.items()}
        payload = {"arm": "TUNE", "note": _ARM_NOTE["TUNE"],
                   "report": {k: _rep.get(k) for k in ("baseline", "best", "delta",
                                                       "per_task_delta", "trials_evaluated",
                                                       "queries", "skipped_queries",
                                                       "active_signals", "scorer")},
                   "candidate_shrunk": _shrink(_rep["best"]["weights"]),
                   "battle_shrunk": _BATTLE_W,
                   "cv": {"mean_heldout": _cv_mean, "paired_net_folds": _net,
                          "folds": len(_cv["candidate"]), "scorer": _sname},
                   "minutes": round((time.time() - _t0) / 60, 1),
                   "started_at": _start, "ended_at": _now(), "session": SESSION,
                   "battle_weights_untouched": True}
        _save("TUNE", payload)
        print(f"   ⚖ ứng viên (shrink 50%): {payload['candidate_shrunk']}")
        print(f"   ⚖ in-sample: baseline {_rep['baseline']['score']:.4f} → best "
              f"{_rep['best']['score']:.4f} (Δ {_rep['delta']:+.4f})")
        print(f"   ⚖ held-out 16 fold (retrieval-only): {_cv_mean} · ứng viên thắng ròng "
              f"{_net:+d} fold so với MẶC ĐỊNH (trọng số trận đã khớp cả 23 câu nên "
              "battle_insample chỉ để tham khảo — audit r88b)")
        if _rep["delta"] <= 0:
            print("   ⚠ tuner KHÔNG tìm được bộ thắng in-sample — ABK+TUNED sẽ tự bỏ")
        if _cv_mean["candidate"] < _cv_mean["default"]:
            print("   ⚠ held-out: ứng viên THUA cả mặc định = overfit — dù bench có thắng "
                  "cũng chỉ là khớp 23 câu")
        return payload

    def _merge(arm):
        parts, hedge = _MERGES[arm]
        _missing = [p for p in parts if p not in _results or _results[p].get("skipped")]
        if _missing:
            print(f"   ⏭ {arm}: THIẾU cánh {_missing} — chạy các cánh đó xong rồi chạy lại cell.")
            return None
        for p in parts:                          # phần trộn phải ĐỦ CSV của nó
            _dir = _camp / f"{p}_run"
            try:
                list(_dir.iterdir())
            except OSError:
                pass
            _stems = {q.stem for q in _dir.glob("query-*.csv")}
            _need = set(_results[p].get("per_query", {}))
            if not _need <= _stems:
                raise RuntimeError(f"[{arm}] {p}_run/ thiếu CSV {sorted(_need - _stems)} "
                                   "— không trộn lượt thiếu")
        _runs = [load_run(_camp / f"{p}_run") for p in parts]
        _merged = rrf_merge_runs(_runs, weights=None, k=60, qa_keep_all_answers=hedge)
        _out = _local / "submissions" / f"camp_{arm}"
        shutil.rmtree(_out, ignore_errors=True)
        write_merged(_merged, _out)
        r = score_run(_out, GT_PATH)
        payload = {**r.to_dict(), "arm": arm, "note": _ARM_NOTE[arm], "parts": parts,
                   "hedge": hedge, "minutes": 0.0, "storm": {}, "degraded": {},
                   "session": SESSION,
                   "sessions_of_parts": {p: _results[p].get("session") for p in parts}}
        _save(arm, payload, _out)
        return payload

    _t_all = time.time()
    # Round-89: cánh thêm sau (vd G38) chạy ở PHIÊN KHÁC với ABK trên Drive →
    # theo luật "cùng phiên" sẽ không bao giờ được xét thắng. Baseline phải
    # cùng phiên: nếu còn cánh bench chưa đo mà ABK trên Drive thuộc phiên
    # khác thì ĐO LẠI ABK; bản cũ xoay sang ABK-prev*.json = thêm một lần đo
    # nhiễu (càng nhiều lần đo, ngưỡng thắng càng thật).
    _abk_raw = None                              # đọc FILE THÔ (audit r89: _done()
    if "ABK" in _arms and (_camp / "ABK.json").exists():   # trả None cả khi CSV lag)
        try:
            _abk_raw = json.loads((_camp / "ABK.json").read_text(encoding="utf-8"))
        except (OSError, ValueError):
            _abk_raw = {}
    _pending = [a for a in _arms if a in _BENCH_ARMS and a != "ABK" and _done(a) is None]
    _reabk = bool(_abk_raw is not None and _abk_raw.get("session") != SESSION and _pending)
    if _reabk:
        print(f"↻ ABK trên Drive thuộc phiên {_abk_raw.get('session')}, còn {_pending} chưa đo "
              "→ đo lại ABK trong phiên này (bản cũ xoay sang ABK-prev*.json khi bản mới ghi xong)")
    try:
        for _arm in _arms:
            _prev = _done(_arm)
            if _arm == "ABK" and _reabk:
                _prev = None                     # baseline phải cùng phiên
            if _prev is not None:                # cánh ĐÃ XONG (phiên nào cũng vậy) → nạp
                print(f"⏭ {_arm}: đã có trên Drive (mean_final={_prev.get('mean_final', '—')}, "
                      f"phiên {_prev.get('session', '?')}) — bỏ qua (xóa lab/campaign/{_arm}.json "
                      "để đo lại)")
                _results[_arm] = _prev
                continue
            # Round-90 (phiên 850660ee): chỉ cánh CHƯA ĐO mới bị chặn khi ABK
            # phiên này hỏng — bản cũ từng chặn cả cánh đã xong, bảng mất baseline.
            if (_reabk and _arm in _BENCH_ARMS and _arm != "ABK"
                    and _results.get("ABK", {}).get("session") != SESSION):
                print(f"⏭ {_arm}: ABK phiên này chưa đo được — cánh bench không có baseline "
                      "để so, bỏ qua (chạy lại cell sau khi ABK đo xong)")
                continue
            print(f"\n▶ CÁNH {_arm} — {_ARM_NOTE[_arm]}   [{_now()}]")
            try:
                if _arm == "TUNE":
                    _tune()
                elif _arm in _MERGES:
                    _merge(_arm)
                else:
                    _p = _bench(_arm)
                    if _p and "mean_final" in _p:
                        print(f"   ⭐ {_arm}: mean_final={_p['mean_final']:.4f} "
                              f"({_p['minutes']} phút, bão={_p['storm'] or 'không'}, "
                              f"suy thoái nhẹ={_p['degraded'] or 'không'}, "
                              f"rớt model={_p.get('model_fallbacks') or 'không'}, "
                              f"hết quota={_p.get('quota_429') or 0}×, "
                              f"tự trả lời={_p.get('declared_share')}, "
                              f"VRAM đỉnh {_p['vram_peak_gib']} GiB)")
            except Exception as _e:   # noqa: BLE001 — một cánh hỏng không được giết cả chiến dịch
                import traceback
                traceback.print_exc()
                print(f"   ❌ CÁNH {_arm} HỎNG — KHÔNG LƯU: {type(_e).__name__}: {_e}\n"
                      "   → các cánh còn lại vẫn chạy; chạy lại cell sau sẽ đo lại cánh này.")
            gc.collect()
            torch.cuda.empty_cache()

        if "ABK" not in _results and _abk_raw and "per_query" in _abk_raw:
            print(f"\n⚠ ABK phiên này KHÔNG đo được — dùng ABK phiên {_abk_raw.get('session')} "
                  "làm baseline cho bảng (cánh đo ở phiên này không được xét thắng)")
            _results["ABK"] = _abk_raw
        # ── Tổng kết: nhiễu đo được, bảng cánh, ma trận từng câu, phán quyết ──
        print("\n" + "═" * 78)
        print(f"📊 TỔNG KẾT CHIẾN DỊCH ({(time.time() - _t_all) / 60:.0f} phút phiên này)")
        _abk = _results.get("ABK")
        _draws = {}                              # các lần đo ABK: phiên này + Drive (chỉ đọc)
        if _abk and "mean_final" in _abk:
            _draws["ABK"] = _abk
        try:
            list((PROJECT / "artifacts" / "lab").iterdir())
        except OSError:
            pass
        for _f in sorted(_camp.glob("ABK-prev*.json")):     # ABK của các phiên trước
            try:
                _d = json.loads(_f.read_text(encoding="utf-8"))
                if "per_query" in _d:
                    _draws[_f.stem] = _d
            except (OSError, ValueError):
                pass
        for _nm in ("bench_full.json", "bench_full-prev.json"):
            _f = PROJECT / "artifacts" / "lab" / _nm
            if _f.exists():
                try:
                    _d = json.loads(_f.read_text(encoding="utf-8"))
                    if _d.get("bench_pack") == "ABK" and "per_query" in _d:
                        _draws["ABK_prev" if _nm == "bench_full.json" else "ABK_prev2"] = _d
                except (OSError, ValueError):
                    pass
        _base = _abk.get("mean_final") if _abk else None
        _num_gt = (_abk or {}).get("num_gt") or len(_qfiles)
        _step = 0.2 / _num_gt
        _means = [d["mean_final"] for d in _draws.values()]
        _noise = (max(_means) - min(_means)) if len(_means) > 1 else 0.0
        _pq = {n: d["per_query"] for n, d in _draws.items()}
        if len(_pq) > 1:
            _a, _b = list(_pq.values())[:2]
            _disagree = sum(1 for s in _a if s in _b
                            and abs(_a[s]["final"] - _b[s]["final"]) > 1e-9)
        else:
            _disagree = None
        _bar = max(2 * _noise, 2 * _step)
        # Round-89: mỗi cánh so Δ với lần đo ABK CÙNG PHIÊN với nó (ABK hoặc
        # ABK-prev*), chỉ gắn "≠phiên" khi không có; thắng ròng vẫn tính trên
        # MỌI lần đo ABK (chặt hơn).
        _abk_by_session = {d.get("session"): d for d in _draws.values() if d.get("session")}
        print(f"Nhiễu ABK: {len(_draws)} lần đo {[round(m, 4) for m in _means]} → nhiễu "
              f"{_noise:.4f}; bậc điểm 0.2/{_num_gt} = {_step:.4f}; "
              f"câu lệch giữa 2 lần đo: {_disagree}")
        if len(_draws) <= 1:
            print("⚠ KHÔNG có bench ABK cũ trên Drive (lab/bench_full*.json) — chỉ 1 lần đo ABK, "
                  "ngưỡng thắng theo bậc điểm 2×0.2/N")
        print(f"Luật thắng: Δ ≥ {_bar:.4f} VÀ thắng ròng ≥ 2 câu so với mọi lần đo ABK "
              f"VÀ cùng phiên với ABK\n")
        print(f"{'cánh':15s} {'mean_final':>10s} {'Δ vs ABK':>9s} {'ròng':>5s} {'phút':>5s} "
              f"{'429/503':>7s} {'suythoái':>8s} {'VRAM':>5s}  ghi chú")
        _wins, _flags = [], {}
        for _arm in _ARM_ORDER:
            _p = _results.get(_arm)
            if _p is None:
                print(f"{_arm:15s} {'—':>10s}  (chưa có)")
                continue
            if _p.get("skipped"):
                print(f"{_arm:15s} {'(bỏ)':>10s}  {_p.get('reason')}")
                continue
            if "mean_final" not in _p:
                _cvm = _p.get("cv", {}).get("mean_heldout", {})
                print(f"{_arm:15s} {'(dò)':>10s} {'':>9s} {'':>5s} {_p.get('minutes', 0):>5}  "
                      f"held-out ứng viên {_cvm.get('candidate')} vs mặc định "
                      f"{_cvm.get('default')} (trận in-sample {_cvm.get('battle_insample')}; "
                      f"ròng {_p.get('cv', {}).get('paired_net_folds', 0):+d} fold) "
                      f"· in-sample Δ {_p.get('report', {}).get('delta', 0.0):+.4f}")
                continue
            _sess = set(_p.get("sessions_of_parts", {}).values()) or {_p.get("session")}
            _own = _abk_by_session.get(next(iter(_sess))) if len(_sess) == 1 else None
            _base_for = _own["mean_final"] if _own else _base
            _d = _p["mean_final"] - _base_for if _base_for is not None else None
            _net = None
            if _pq and _arm != "ABK":
                _net = 0
                for _s, _qs in _p["per_query"].items():
                    _vals = [q[_s]["final"] for q in _pq.values() if _s in q]
                    if not _vals:
                        continue
                    _net += (1 if _qs["final"] > max(_vals) + 1e-9
                             else -1 if _qs["final"] < min(_vals) - 1e-9 else 0)
            _fl = []
            if _arm != "ABK" and _own is None:
                _fl.append("≠phiên")
            elif _own is not None and _own is not _abk:
                _fl.append(f"so với ABK phiên {_own.get('session')}")
            if _p.get("degraded"):
                _fl.append("suy thoái nhẹ")
            _dm = _p.get("declared_model")
            _sh = _p.get("declared_share")
            if _dm and _sh is not None:           # round-92: ≥90% tự trả lời mới được xét thắng
                if _sh < 0.9:
                    _fl.append(f"rớt model {1 - _sh:.0%}")
            elif _dm and _p.get("model_fallbacks", {}).get(_dm):
                _fl.append(f"rớt model {_p['model_fallbacks'][_dm]}×")
            if _p.get("quota_429"):
                _fl.append(f"hết quota {_p['quota_429']}×")   # QA đo bằng model cứu viện
            if _arm == "ABK+TUNED" and "TUNE" in _results:
                _cvm = _results["TUNE"].get("cv", {}).get("mean_heldout", {})
                if _cvm and _cvm.get("candidate", 0) < _cvm.get("default", 0):
                    _fl.append("thua held-out")
            _flags[_arm] = _fl
            _win = (_arm != "ABK" and _d is not None and _d >= _bar - 1e-9
                    and (_net is None or _net >= 2) and "≠phiên" not in _fl
                    and "thua held-out" not in _fl
                    and not any(f.startswith(("rớt model", "hết quota")) for f in _fl))
            if _win:
                _wins.append(_arm)
            _s = _p.get("storm", {})
            _st = f"{_s.get('429', 0)}/{_s.get('503', 0)}"
            print(f"{_arm:15s} {_p['mean_final']:>10.4f} "
                  f"{(f'{_d:+.4f}' if _d is not None else '—'):>9s} "
                  f"{(f'{_net:+d}' if _net is not None else '—'):>5s} {_p.get('minutes', 0):>5} "
                  f"{_st:>7s} {sum(_p.get('degraded', {}).values()):>8d} "
                  f"{_p.get('vram_peak_gib', '—'):>5}  {'🏆 ' if _win else ''}"
                  f"{_p['note']}{(' ⚠ ' + ', '.join(_fl)) if _fl else ''}")
            if _p.get("by_task"):
                print(f"{'':15s} theo task: "
                      + ", ".join(f"{t}={v:.3f}" for t, v in sorted(_p["by_task"].items())))
        _cols = ([a for a in _ARM_ORDER if a in _results and "per_query" in _results[a]]
                 + [n for n in _pq if n != "ABK"])
        _allpq = {**{a: _results[a]["per_query"] for a in _cols if a in _results}, **_pq}
        _stems = sorted({s for q in _allpq.values() for s in q})
        if _stems:
            print("\nĐiểm từng câu (hàng = câu, cột = cánh; ABK_prev = bench ABK cũ trên Drive):")
            print(f"{'câu':22s} {'task':5s} " + " ".join(f"{c[:9]:>9s}" for c in _cols))
            for _s in _stems:
                _task = next((q[_s].get("task", "") for q in _allpq.values() if _s in q), "")
                print(f"{_s:22s} {_task:5s} " + " ".join(
                    f"{_allpq[c].get(_s, {}).get('final', float('nan')):>9.3f}" for c in _cols))
        print("\n🏆 THẮNG ABK theo luật trên:", _wins or "không cánh nào — trận giữ ABK")
        _write_json(_camp / "campaign_summary.json", {
            "base_abk": _base, "abk_draws": {n: d["mean_final"] for n, d in _draws.items()},
            "noise": _noise, "step": _step, "win_bar": _bar, "wins": _wins, "flags": _flags,
            "session": SESSION, "gpu": _GPU,
            "arms": {a: {k: v for k, v in p.items() if k != "per_query"}
                     for a, p in _results.items()},
            "per_query": _allpq})
        shutil.copy2(_camp / "campaign_summary.json", _camp / f"campaign_summary-{SESSION}.json")
        print(f"📁 Drive: {_camp} — gửi Claude campaign_summary.json (+ bản theo phiên "
              f"campaign_summary-{SESSION}.json) + output cell này.")
    finally:
        logging.getLogger().removeHandler(_storm)

In [ ]:
# ── L6 · 🫀 Giữ phiên sống sau khi Lab xong (bấm ⏹ của ô này để dừng) ──
# Round-46: phiên Lab từng bị Colab thu hồi vì "không hoạt động". Kernel bận
# chạy ô này = hoạt động. Kết quả các stage đã được LƯU THẲNG LÊN DRIVE ngay
# khi có (bench_full.json / best_weights.json / lane_ab.json / lab_full/),
# nên dù phiên chết cũng không mất bài — ô này chỉ giữ máy ảo cho bạn quay
# lại chạy thêm stage. GIỮ TAB TRÌNH DUYỆT MỞ trong lúc Lab chạy.
import time as _tm
print("🫀 Lab watchkeeper — phiên được giữ sống.")
try:
    _n = 0
    while True:
        _tm.sleep(30)
        _n += 1
        if _n % 10 == 0:
            print(f"🫀 {_tm.strftime('%H:%M')} phiên sống")
except KeyboardInterrupt:
    print("⏹ Dừng — phiên sẽ tính là nhàn rỗi từ giờ.")